# 단계 08. 화소 단위 AE·S2-TSFM 비교와 최종 변화지도

논문 3.7절(서비스 입력), 4.3절(Figure 6a 최종지도 연도) · 절차서: `forest-change-agent/EXPERIMENTS.md` 순서 8

입력: `Chuncheon_labels_S2_20m.tif`·`Chuncheon_labels_AE_10m.tif`(`prep/build_training_label_rasters.py`), `Chuncheon_forestmask_*.tif`(`prep/build_forest_mask.py`), `Chuncheon_convlabels_AE_10m.tif`(`prep/build_conversion_labels.py`), 단계 01 래스터

순서: 라벨 격자 맞춤 → S2-TSFM(54밴드)·AE 분류기 비교 → 0벡터 화소 진단·수정 → 산림 마스크 → 최소면적 → 최종지도 → 전용 고시 라벨 검증·보강

산출: `compare_runs/`, `final_change_2019_2025/`

모듈: `step05_s2_tsfm.py`, `step08_ae_change_classifier.py`, `step08_final_change_map.py`, `step08_conv_label_experiment.py`

> 셀 출력은 원 실행 기록입니다(원본 노트북: scratchpad (4).ipynb 셀 19~27 + scratchpad (5).ipynb 전체). 당시 이름(Mini-GFM, `mini_gfm_ts.py`, `GEE_Export/mini_gfm_chuncheon` 등)은 코드에서 S2-TSFM·단계 번호 이름으로 바꿨고, 출력 문구는 실행 당시 그대로입니다.


In [ ]:
# [08-0] Drive 연결 (이 노트북만 단독 실행할 때)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
%%writefile step05_s2_tsfm.py
#!/usr/bin/env python3
# coding: utf-8
# [colab 정리본] 원본: s2_tsfm/s2_tsfm.py — 파일명·모듈명만 실험 단계 번호에 맞춤 (계산 로직 동일)
"""S2-TSFM 다연도판: 춘천 Sentinel-2 42밴드(2019~2025, 7년 × 6밴드) 산림변화 탐지.

이전 3년판(18밴드)의 구조를 N년으로 일반화하고, 자기지도 사전학습 뒤 라벨로 변화 분류기를 학습한다.
  1) 입력: 연도별 [NDVI, NDWI, NDBI, NDMI, Slope, Elevation] × N  →  논리 밴드 [NDVI, NDWI, NDMI] × N + Slope + Elevation
     (NDBI = −NDMI 이므로 제외, 지형은 첫 해 것만 사용 — 이전 3년판과 동일)
  2) 사전학습: 연도 공유 CNN 인코더 + 시간 결합 디코더, 공간 블록 마스크 복원(L1). 연도 전체를 가리는 마스크도 섞어
     시간 문맥으로 복원하도록 함. 학습/검증은 512 px 공간 블록 단위로 분리
  3) 추론(change_scores.tif): 연속 연도 임베딩 거리 N−1개, 첫해→끝해 거리, 최대 연속 거리, 최대 변화 연도,
     ΔNDVI·ΔNDMI(끝−첫), 최대 NDVI 감소량과 그 연도
  4) 변화 분류기: 라벨 래스터의 학습 라벨로 화소 분류기 학습
     - 새 형식(build_training_label_rasters.py): 경영실적·임상도 합의로 거른 TRAIN_P1/P2 (--period 로 구간 선택)
     - 이전 형식(make_s2_label_raster.py): 임상도 2019→2025 변화 FC_TYPE
     (HistGradientBoosting, 공간 블록 홀드아웃). 특징 세트 비교: 지수만 / 지수+GFM 임베딩
  5) 평가: AUROC·AP(변화 여부, 벌채), 국유림 산림사업실적(MGMT, 춘천_임소반_산림사업실적.gpkg 2019~2025)과의 일치,
     변화 연도 ±1년 정확도
     → change_class.tif (0 무변화, 1 벌채·갱신, 2 산지전용, 3 산림 증가) + change_prob.tif

Colab:
  %run step05_s2_tsfm.py all --input /content/data/Chuncheon_S2_2019_2025_42Bands.tif \
       --labels /content/data/Chuncheon_labels_S2_20m.tif --period P2 --outdir /content/runs/chuncheon --epochs 30
"""
from __future__ import annotations

import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import rasterio
import torch
from torch import nn

NODATA = -9999.0
FIRST_YEAR = 2019
FC_NAMES = {0: "no_change", 1: "harvest", 2: "conversion", 3: "gain"}
# 합의 라벨(CONS) → 실적 기준 평가 라벨(MGMT): 0 실적없음, 1 수확·갱신 실적, 2 솎아베기 실적, 3 피해·기타 실적
CONS_TO_MGMT = {0: 0, 2: 0, 1: 1, 3: 1, 4: 2, 5: 3}


def read_labels(path, period):
    """라벨 래스터 → {FC_TYPE, FC_YEAR, MGMT, SOURCE}.
    새 형식(build_training_label_rasters.py: TRAIN_P1/P2, YEAR_P*, CONS_P*, SOURCE)과
    이전 형식(make_s2_label_raster.py: FC_TYPE, FC_YEAR, MGMT, MGMT_YEAR)을 모두 읽는다."""
    with rasterio.open(path) as lb:
        bands = {d: lb.read(i + 1) for i, d in enumerate(lb.descriptions)}
        size = (lb.width, lb.height)
    if f"TRAIN_{period}" in bands:
        cons = bands[f"CONS_{period}"]
        mgmt = np.full(cons.shape, 255, np.uint8)
        for c, m in CONS_TO_MGMT.items():
            mgmt[cons == c] = m
        labels = {"FC_TYPE": bands[f"TRAIN_{period}"], "FC_YEAR": bands[f"YEAR_{period}"],
                  "MGMT": mgmt, "SOURCE": bands["SOURCE"]}
        fmt = f"새 형식 {period} (TRAIN·YEAR·CONS·SOURCE)"
    else:
        labels = {k: bands[k] for k in ("FC_TYPE", "FC_YEAR", "MGMT")}
        labels["SOURCE"] = np.where(bands["MGMT"] != 255, 1, 2).astype(np.uint8)
        fmt = "이전 형식 (FC_TYPE·FC_YEAR·MGMT)"
    return labels, size, fmt


# ------------------------------------------------------------------ 입력

def load_stack(path):
    """42밴드 → 논리 배열 [3N+2, H, W] (float32, 무효 NaN) + 유효 마스크."""
    with rasterio.open(path) as ds:
        if ds.count % 6:
            raise ValueError(f"6의 배수 밴드여야 함: {ds.count}")
        n = ds.count // 6
        desc = ds.descriptions
        expected = ["NDVI_", "NDWI_", "NDBI_", "NDMI_", "Slope_", "Elevation_"] * n
        if all(desc):
            bad = [(i + 1, d) for i, (d, e) in enumerate(zip(desc, expected)) if not str(d).startswith(e)]
            if bad:
                raise ValueError(f"밴드 순서가 GEE 내보내기와 다름: {bad[:5]}")
            years = [int(desc[6 * y].split("_")[1]) for y in range(n)]
        else:
            years = list(range(FIRST_YEAR, FIRST_YEAR + n))
        idx = [6 * y + b + 1 for y in range(n) for b in (0, 1, 3)] + [5, 6]
        arr = ds.read(idx, masked=True).astype("float32").filled(np.nan)
        meta = {"profile": ds.profile.copy(), "years": years}
    valid = np.isfinite(arr).all(axis=0)
    valid &= (np.abs(arr[:3 * n]) <= 1.05).all(axis=0)
    valid &= (arr[3 * n] >= 0) & (arr[3 * n] <= 90) & (arr[3 * n + 1] > -500)
    return arr, valid, n, meta


def normalize(arr, valid, n, stats):
    x = np.zeros_like(arr, dtype=np.float32)
    x[:3 * n] = np.clip(arr[:3 * n], -1, 1) * 2.0
    x[3 * n] = np.clip((arr[3 * n] - stats["slope_mean"]) / stats["slope_std"], -5, 5)
    x[3 * n + 1] = np.clip((arr[3 * n + 1] - stats["elev_mean"]) / stats["elev_std"], -5, 5)
    x[:, ~valid] = 0.0
    x[~np.isfinite(x)] = 0.0
    return x


def block_split(h, w, block, seed, val_frac=0.2):
    """512 px 공간 블록 단위 학습/검증(=홀드아웃) 분할 마스크. True = 홀드아웃 블록."""
    by, bx = math.ceil(h / block), math.ceil(w / block)
    rng = np.random.default_rng(seed)
    hold = rng.random((by, bx)) < val_frac
    return np.kron(hold, np.ones((block, block), bool))[:h, :w]


# ------------------------------------------------------------------ 모델 (3년판 TemporalMaskedAE 의 N년 일반화)

class ResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), nn.GroupNorm(4, c), nn.GELU(),
                                 nn.Conv2d(c, c, 3, padding=1), nn.GroupNorm(4, c))
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class TemporalMaskedAE(nn.Module):
    def __init__(self, n_years, dim=24):
        super().__init__()
        self.n = n_years
        self.encoder = nn.Sequential(nn.Conv2d(6, dim, 3, padding=1), nn.GELU(),
                                     ResidualBlock(dim), ResidualBlock(dim))
        self.decoder = nn.Sequential(nn.Conv2d(n_years * dim, dim * 2, 3, padding=1), nn.GELU(),
                                     ResidualBlock(dim * 2), nn.Conv2d(dim * 2, 3 * n_years, 1))

    def encode_years(self, x, mask=None):
        n = self.n
        topo = x[:, 3 * n:3 * n + 2]
        emb = []
        for y in range(n):
            idx = x[:, 3 * y:3 * y + 3]
            m = torch.zeros_like(idx[:, :1]) if mask is None else mask[:, y:y + 1]
            emb.append(self.encoder(torch.cat((idx * (1 - m), topo, m), dim=1)))
        return emb

    def forward(self, x, mask):
        return self.decoder(torch.cat(self.encode_years(x, mask), dim=1))


def make_mask(b, n, hw, ratio, block, year_drop, device):
    h, w = hw
    small = torch.rand(b, n, math.ceil(h / block), math.ceil(w / block), device=device) < ratio
    m = nn.functional.interpolate(small.float(), size=(h, w), mode="nearest")
    drop = (torch.rand(b, n, 1, 1, device=device) < year_drop).float()  # 연도 전체 가리기
    return torch.maximum(m, drop)


def masked_l1(pred, x, valid, mask, n):
    use = mask.repeat_interleave(3, dim=1) * valid
    return (torch.abs(pred - x[:, :3 * n]) * use).sum() / use.sum().clamp_min(1.0)


# ------------------------------------------------------------------ 1. 사전학습

def sample_patches(x, valid, region_ok, patch, k, rng):
    h, w = valid.shape
    out_x, out_v = [], []
    tries = 0
    while len(out_x) < k and tries < k * 50:
        tries += 1
        r, c = rng.integers(0, h - patch), rng.integers(0, w - patch)
        if region_ok[r:r + patch, c:c + patch].mean() < 0.9 or valid[r:r + patch, c:c + patch].mean() < 0.5:
            continue
        xx, vv = x[:, r:r + patch, c:c + patch], valid[r:r + patch, c:c + patch]
        rot = rng.integers(4)
        xx, vv = np.rot90(xx, rot, axes=(1, 2)), np.rot90(vv, rot)
        if rng.random() < 0.5:
            xx, vv = xx[:, :, ::-1], vv[:, ::-1]
        out_x.append(np.ascontiguousarray(xx))
        out_v.append(np.ascontiguousarray(vv[None]))
    return torch.from_numpy(np.stack(out_x)), torch.from_numpy(np.stack(out_v).astype("float32"))


def pretrain(args, x, valid, n, hold, device, outdir):
    torch.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)
    model = TemporalMaskedAE(n, args.dim).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, args.epochs)
    val_x, val_v = sample_patches(x, valid, hold, args.patch, 32, np.random.default_rng(0))
    best, history = float("inf"), []
    for ep in range(1, args.epochs + 1):
        model.train()
        t, losses = time.time(), []
        for _ in range(args.steps):
            xb, vb = sample_patches(x, valid, ~hold, args.patch, args.batch, rng)
            xb, vb = xb.to(device), vb.to(device)
            m = make_mask(xb.shape[0], n, xb.shape[-2:], args.mask_ratio, args.mask_block, args.year_drop, device)
            loss = masked_l1(model(xb, m), xb, vb, m, n)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sched.step()
        model.eval()
        with torch.no_grad():
            g = torch.Generator(device=device).manual_seed(1)
            vl = []
            for i in range(0, len(val_x), args.batch):
                xb, vb = val_x[i:i + args.batch].to(device), val_v[i:i + args.batch].to(device)
                small = torch.rand(xb.shape[0], n, math.ceil(args.patch / args.mask_block),
                                   math.ceil(args.patch / args.mask_block), device=device, generator=g)
                m = nn.functional.interpolate((small < args.mask_ratio).float(), size=xb.shape[-2:], mode="nearest")
                vl.append(masked_l1(model(xb, m), xb, vb, m, n).item())
        tr, va = float(np.mean(losses)), float(np.mean(vl))
        history.append({"epoch": ep, "train_mae": tr, "val_mae": va})
        flag = ""
        if va < best:
            best = va
            torch.save({"model_state": model.state_dict(), "n_years": n, "dim": args.dim,
                        "stats": args.stats, "best_val_masked_mae": best}, outdir / "best_model.pt")
            flag = " *saved"
        print(f"[pretrain] epoch {ep:03d}/{args.epochs} masked MAE train {tr:.4f} val {va:.4f} "
              f"({time.time() - t:.0f}s){flag}", flush=True)
    (outdir / "history.json").write_text(json.dumps(history, indent=2))
    return best


# ------------------------------------------------------------------ 2. 추론 (전체 영상 타일 처리)

def infer(model, x, valid, n, years, tile, halo, device):
    """반환: scores[dict 이름→H×W], emb_delta[dim,H,W](끝해−첫해 임베딩)."""
    _, h, w = x.shape
    names = [f"emb_d{years[i]}_{years[i + 1]}" for i in range(n - 1)] + [
        "emb_first_last", "emb_max_consec", "emb_max_year", "dNDVI", "dNDMI", "ndvi_max_drop", "ndvi_drop_year"]
    scores = {k: np.full((h, w), np.nan, np.float32) for k in names}
    dim = model.encoder[0].out_channels
    emb_delta = np.zeros((dim, h, w), np.float16)
    xp = np.pad(x, ((0, 0), (halo, halo), (halo, halo)))
    model.eval()
    for r in range(0, h, tile):
        for c in range(0, w, tile):
            th, tw = min(tile, h - r), min(tile, w - c)
            xt = torch.from_numpy(xp[None, :, r:r + th + 2 * halo, c:c + tw + 2 * halo]).to(device)
            with torch.inference_mode():
                emb = [e[0, :, halo:halo + th, halo:halo + tw] for e in model.encode_years(xt)]
                d = torch.stack([(emb[i + 1] - emb[i]).square().mean(0).sqrt() for i in range(n - 1)])
                fl = (emb[-1] - emb[0]).square().mean(0).sqrt()
            d, fl = d.cpu().numpy(), fl.cpu().numpy()
            win = (slice(r, r + th), slice(c, c + tw))
            for i in range(n - 1):
                scores[names[i]][win] = d[i]
            scores["emb_first_last"][win] = fl
            scores["emb_max_consec"][win] = d.max(0)
            scores["emb_max_year"][win] = np.asarray(years[1:])[d.argmax(0)]
            emb_delta[(slice(None), *win)] = (emb[-1] - emb[0]).cpu().numpy()
    ndvi = x[0:3 * n:3] / 2.0
    ndmi = x[2:3 * n:3] / 2.0
    scores["dNDVI"] = ndvi[-1] - ndvi[0]
    scores["dNDMI"] = ndmi[-1] - ndmi[0]
    steps = np.diff(ndvi, axis=0)
    scores["ndvi_max_drop"] = -steps.min(0)
    scores["ndvi_drop_year"] = np.asarray(years[1:], np.float32)[steps.argmin(0)]
    for k in scores:
        scores[k][~valid] = np.nan
    return scores, emb_delta


def write_tif(path, profile, bands: dict, dtype="float32", nodata=NODATA):
    p = profile.copy()
    p.update(driver="GTiff", count=len(bands), dtype=dtype, nodata=nodata, compress="deflate",
             tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(path, "w", **p) as dst:
        for i, (k, v) in enumerate(bands.items(), 1):
            dst.write(np.where(np.isfinite(v), v, nodata).astype(dtype), i)
            dst.set_band_description(i, k)


# ------------------------------------------------------------------ 3. 분류기·평가

def features(x, n, scores, emb_delta, with_gfm):
    """화소 특징 행렬 [H*W, F]. 지수 기반(원시 지수·연속 차분·지형) + 선택적으로 GFM 임베딩 특징."""
    ndvi, ndwi, ndmi = x[0:3 * n:3] / 2, x[1:3 * n:3] / 2, x[2:3 * n:3] / 2
    feats = [ndvi, ndwi, ndmi, np.diff(ndvi, axis=0), np.diff(ndmi, axis=0), x[3 * n:3 * n + 2],
             np.stack([scores["dNDVI"], scores["dNDMI"], scores["ndvi_max_drop"]])]
    names = ([f"ndvi{i}" for i in range(n)] + [f"ndwi{i}" for i in range(n)] + [f"ndmi{i}" for i in range(n)]
             + [f"dndvi{i}" for i in range(n - 1)] + [f"dndmi{i}" for i in range(n - 1)]
             + ["slope", "elev", "dNDVI", "dNDMI", "ndvi_max_drop"])
    if with_gfm:
        dist = np.stack([v for k, v in scores.items() if k.startswith("emb_") and k != "emb_max_year"])
        feats += [dist, emb_delta.astype(np.float32)]
        names += [k for k in scores if k.startswith("emb_") and k != "emb_max_year"]
        names += [f"embdelta{i}" for i in range(emb_delta.shape[0])]
    f = np.concatenate([a.reshape(a.shape[0], -1) for a in feats]).T
    return f.astype(np.float32), names


def auc_ap(y, s):
    from sklearn.metrics import average_precision_score, roc_auc_score
    ok = np.isfinite(s)
    if ok.sum() == 0 or len(np.unique(y[ok])) < 2:
        return float("nan"), float("nan")
    return float(roc_auc_score(y[ok], s[ok])), float(average_precision_score(y[ok], s[ok]))


def evaluate_scores(scores, labels, hold):
    """비지도 점수의 변화 탐지력 (홀드아웃 블록 화소)."""
    fc, mg, nat = labels["FC_TYPE"], labels["MGMT"], labels["SOURCE"] == 1
    tasks = {
        "임상도 변화여부(벌채·전용·증가 vs 무변화)": (np.isin(fc, [1, 2, 3]), fc == 0),
        "국유림 검증라벨 변화여부(실적·임상도 합의)": (np.isin(fc, [1, 2, 3]) & nat, (fc == 0) & nat),
        "임상도 벌채·갱신 vs 무변화": (fc == 1, fc == 0),
        "국유림 실적 수확·갱신 vs 실적없음": (mg == 1, mg == 0),
        "국유림 실적 솎아베기 vs 실적없음": (mg == 2, mg == 0),
    }
    cands = {"GFM emb_first_last": scores["emb_first_last"], "GFM emb_max_consec": scores["emb_max_consec"],
             "지수 |ΔNDVI|": np.abs(scores["dNDVI"]), "지수 −ΔNDVI": -scores["dNDVI"],
             "지수 최대 NDVI 감소": scores["ndvi_max_drop"]}
    rows = []
    for task, (pos, neg) in tasks.items():
        m = hold & (pos | neg)
        y = pos[m].astype(int)
        for name, s in cands.items():
            auc, ap = auc_ap(y, s[m])
            rows.append({"task": task, "score": name, "n_pos": int(y.sum()), "n_neg": int((1 - y).sum()),
                         "prevalence": round(float(y.mean()), 4) if len(y) else None,
                         "auroc": round(auc, 4), "ap": round(ap, 4)})
    return rows


def train_classifier(F, names, labels, hold, seed, max_per_class=60000):
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
    fc = labels["FC_TYPE"].ravel()
    tr = (~hold.ravel()) & (fc != 255) & np.isfinite(F).all(1)
    te = hold.ravel() & (fc != 255) & np.isfinite(F).all(1)
    rng = np.random.default_rng(seed)
    idx = np.concatenate([rng.permutation(np.where(tr & (fc == k))[0])[:max_per_class] for k in FC_NAMES])
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=63,
                                         l2_regularization=1.0, class_weight="balanced", random_state=seed)
    clf.fit(F[idx], fc[idx])
    pred = clf.predict(F[te])
    proba = clf.predict_proba(F[te])
    changed_true = np.isin(fc[te], [1, 2, 3]).astype(int)
    p_changed = proba[:, [list(clf.classes_).index(k) for k in (1, 2, 3)]].sum(1)
    auc, ap = auc_ap(changed_true, p_changed)
    nat = labels["SOURCE"].ravel()[te] == 1
    v_auc, v_ap = auc_ap(changed_true[nat], p_changed[nat])
    rep = classification_report(fc[te], pred, labels=list(FC_NAMES), target_names=list(FC_NAMES.values()),
                                output_dict=True, zero_division=0)
    metrics = {"n_train": int(len(idx)), "n_test": int(te.sum()),
               "balanced_accuracy": round(float(balanced_accuracy_score(fc[te], pred)), 4),
               "macro_f1": round(float(f1_score(fc[te], pred, average="macro")), 4),
               "changed_auroc": round(auc, 4), "changed_ap": round(ap, 4),
               "verified_changed_auroc": round(v_auc, 4), "verified_changed_ap": round(v_ap, 4),
               "n_test_verified": int(nat.sum()),
               "per_class": {k: {m: round(v[m], 4) for m in ("precision", "recall", "f1-score", "support")}
                             for k, v in rep.items() if k in FC_NAMES.values()}}
    return clf, metrics


def year_accuracy(scores, labels, cls_map, hold):
    """임상도 갱신년도(약한 라벨)와 GFM 최대 변화 연도 비교 (변화로 분류·라벨된 홀드아웃 화소)."""
    fy = labels["FC_YEAR"].astype(float)
    fy[fy == 255] = np.nan
    fy += 2000
    m = hold & np.isfinite(fy) & np.isin(labels["FC_TYPE"], [1, 2, 3]) & np.isfinite(scores["emb_max_year"])
    out = {"n": int(m.sum())}
    for k in ("emb_max_year", "ndvi_drop_year"):
        diff = np.abs(scores[k][m] - fy[m])
        out[k] = {"exact": round(float((diff == 0).mean()), 4), "within_1yr": round(float((diff <= 1).mean()), 4)}
    return out


def mgmt_agreement(cls_map, labels):
    """국유림 산림사업실적(2019~2025) 소반 화소에서 모델이 변화(1~3)로 본 비율."""
    mg = labels["MGMT"]
    return {name: round(float(np.isin(cls_map[mg == v], [1, 2, 3]).mean()), 4) if (mg == v).any() else None
            for v, name in ((1, "수확·갱신 실적 소반"), (2, "솎아베기 실적 소반"),
                            (3, "기타(시설·소득·피해복구) 실적 소반"), (0, "실적 없는 소반"))}


# ------------------------------------------------------------------ main

def main():
    p = argparse.ArgumentParser()
    p.add_argument("mode", choices=["all", "pretrain", "evaluate"])
    p.add_argument("--input", required=True)
    p.add_argument("--labels", required=True)
    p.add_argument("--period", choices=["P1", "P2"], default="P2",
                   help="새 형식 라벨의 비교 구간: P1 2013→2019, P2 2019→2025 (이전 형식이면 무시)")
    p.add_argument("--outdir", default="runs/chuncheon")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--steps", type=int, default=100)
    p.add_argument("--batch", type=int, default=8)
    p.add_argument("--patch", type=int, default=128)
    p.add_argument("--dim", type=int, default=24)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--mask-ratio", type=float, default=0.35)
    p.add_argument("--mask-block", type=int, default=16)
    p.add_argument("--year-drop", type=float, default=0.15, help="연도 전체를 가릴 확률")
    p.add_argument("--block", type=int, default=512, help="공간 홀드아웃 블록 크기(px)")
    p.add_argument("--tile", type=int, default=512)
    p.add_argument("--halo", type=int, default=16)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--device", choices=["auto", "cpu", "cuda"], default="auto")
    args, _ = p.parse_known_args()

    device = torch.device("cuda" if args.device == "auto" and torch.cuda.is_available() else
                          ("cpu" if args.device == "auto" else args.device))
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    random.seed(args.seed)
    np.random.seed(args.seed)

    t0 = time.time()
    arr, valid, n, meta = load_stack(args.input)
    years = meta["years"]
    h, w = valid.shape
    print(f"[입력] {Path(args.input).name}: {n}년 {years}, {w}×{h} px, 유효 {valid.mean():.1%}, device {device}")
    vs = valid
    args.stats = {"slope_mean": float(np.nanmean(arr[3 * n][vs])), "slope_std": float(max(1, np.nanstd(arr[3 * n][vs]))),
                  "elev_mean": float(np.nanmean(arr[3 * n + 1][vs])), "elev_std": float(max(1, np.nanstd(arr[3 * n + 1][vs])))}
    x = normalize(arr, valid, n, args.stats)
    del arr
    hold = block_split(h, w, args.block, args.seed)
    labels, lsize, fmt = read_labels(args.labels, args.period)
    if lsize != (w, h):
        raise ValueError(f"라벨 래스터 격자 {lsize} 가 입력 {(w, h)} 과 다름 — build_training_label_rasters.py 로 다시 만드세요")
    fc = labels["FC_TYPE"]
    print(f"[라벨] {fmt}: " + ", ".join(f"{v} {int((fc == k).sum()):,}" for k, v in FC_NAMES.items())
          + f" (국유림 검증 {int(((fc != 255) & (labels['SOURCE'] == 1)).sum()):,} / 국유림 밖 "
          f"{int(((fc != 255) & (labels['SOURCE'] == 2)).sum()):,})")
    print(f"[분할] 홀드아웃 블록 화소 {hold[valid].mean():.1%} (블록 {args.block}px)")

    ckpt = outdir / "best_model.pt"
    if args.mode in ("all", "pretrain"):
        best = pretrain(args, x, valid, n, hold, device, outdir)
        print(f"[pretrain] 완료 best val masked MAE {best:.4f}")
        if args.mode == "pretrain":
            return
    state = torch.load(ckpt, map_location="cpu", weights_only=True)
    model = TemporalMaskedAE(state["n_years"], state["dim"])
    model.load_state_dict(state["model_state"])
    model.to(device)

    print("[infer] 전체 영상 변화점수 계산 …", flush=True)
    scores, emb_delta = infer(model, x, valid, n, years, args.tile, args.halo, device)
    write_tif(outdir / "change_scores.tif", meta["profile"], scores)

    print("[eval] 비지도 변화점수 평가 …", flush=True)
    score_rows = evaluate_scores(scores, labels, hold)

    results = {}
    for tag, with_gfm in (("지수만", False), ("지수+GFM", True)):
        F, names = features(x, n, scores, emb_delta, with_gfm)
        clf, met = train_classifier(F, names, labels, hold, args.seed)
        met["n_features"] = len(names)
        results[tag] = met
        print(f"[classifier] {tag}: 특징 {len(names)}개, balanced acc {met['balanced_accuracy']}, "
              f"macro F1 {met['macro_f1']}, 변화 AUROC {met['changed_auroc']}, AP {met['changed_ap']} | "
              f"국유림 검증 AUROC {met['verified_changed_auroc']}, AP {met['verified_changed_ap']}", flush=True)
        if with_gfm:
            ok = valid.ravel() & np.isfinite(F).all(1)
            prob = np.full((len(FC_NAMES), h * w), np.nan, np.float32)
            prob[:, ok] = clf.predict_proba(F[ok]).T
            cls_map = np.full(h * w, 255, np.uint8)
            cls_map[ok] = clf.classes_[prob[:, ok].argmax(0)]
            cls_map, prob = cls_map.reshape(h, w), prob.reshape(len(FC_NAMES), h, w)
            write_tif(outdir / "change_class.tif", meta["profile"], {"change_class": cls_map}, dtype="uint8", nodata=255)
            write_tif(outdir / "change_prob.tif", meta["profile"],
                      {f"p_{FC_NAMES[k]}": prob[i] for i, k in enumerate(clf.classes_)})
            import joblib
            joblib.dump({"clf": clf, "features": names, "classes": FC_NAMES}, outdir / "change_classifier.joblib")
        del F

    report = {"input": Path(args.input).name, "years": years, "pretrain": json.loads((outdir / "history.json").read_text())[-1]
              if (outdir / "history.json").exists() else None,
              "unsupervised_scores": score_rows, "classifier": results,
              "change_year": year_accuracy(scores, labels, cls_map, hold),
              "mgmt_agreement_pred_changed_share": mgmt_agreement(cls_map, labels),
              "class_area_ha": {FC_NAMES[k]: round(float((cls_map == k).sum()) * 0.04, 1) for k in FC_NAMES},
              "elapsed_min": round((time.time() - t0) / 60, 1)}
    (outdir / "report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    print("\n[비지도 점수 AUROC / AP — 홀드아웃 블록]")
    for r in score_rows:
        print(f"  {r['task']:<32} {r['score']:<22} AUROC {r['auroc']:.3f}  AP {r['ap']:.3f}  (양성 {r['n_pos']:,} / 음성 {r['n_neg']:,})")
    print("[변화 연도 ±1년 정확도]", report["change_year"])
    print("[국유림 실적 소반에서 변화로 분류된 비율]", report["mgmt_agreement_pred_changed_share"])
    print("[분류 면적 ha]", report["class_area_ha"])
    print(f"[완료] {report['elapsed_min']}분, 결과: {outdir}")


if __name__ == "__main__":
    main()


In [41]:
# [08-1] 새 학습 라벨 업로드: Chuncheon_labels_S2_20m.tif + Chuncheon_labels_AE_10m.tif
#        (C:\src\ForestChangeGeoAI\output 에서 두 파일을 함께 선택) → /content/data + Drive/GEE_Export 에 저장
import os, shutil
from google.colab import files
names = ['Chuncheon_labels_S2_20m.tif', 'Chuncheon_labels_AE_10m.tif']
drive_dir = '/content/drive/MyDrive/GEE_Export'
os.makedirs('/content/data', exist_ok=True)
missing = [n for n in names if not os.path.exists(f'{drive_dir}/{n}')]
if missing:
    print('업로드할 파일:', missing)
    for fn in files.upload():
        shutil.move(fn, f'/content/data/{fn}')
        shutil.copy(f'/content/data/{fn}', f'{drive_dir}/{fn}')  # 런타임이 바뀌어도 Drive에서 다시 복사
for n in names:
    if not os.path.exists(f'/content/data/{n}') and os.path.exists(f'{drive_dir}/{n}'):
        shutil.copy(f'{drive_dir}/{n}', f'/content/data/{n}')
import rasterio
for n in names:
    with rasterio.open(f'/content/data/{n}') as ds:
        print(f'{n}: {ds.width}×{ds.height}, {ds.res[0]:.0f} m, {ds.crs}, 밴드 {ds.descriptions}')

업로드할 파일: ['Chuncheon_labels_S2_20m.tif', 'Chuncheon_labels_AE_10m.tif']


Saving Chuncheon_labels_AE_10m.tif to Chuncheon_labels_AE_10m.tif
Saving Chuncheon_labels_S2_20m.tif to Chuncheon_labels_S2_20m.tif
Chuncheon_labels_S2_20m.tif: 2324×2275, 20 m, EPSG:32652, 밴드 ('TRAIN_P1', 'TRAIN_P2', 'YEAR_P1', 'YEAR_P2', 'CONS_P1', 'CONS_P2', 'SOURCE')
Chuncheon_labels_AE_10m.tif: 4648×4550, 10 m, EPSG:32652, 밴드 ('TRAIN_P1', 'TRAIN_P2', 'YEAR_P1', 'YEAR_P2', 'CONS_P1', 'CONS_P2', 'SOURCE')


In [42]:
# [08-2] 라벨 격자 ↔ 실제 영상 격자 일치 확인 (AlphaEarth 10 m, Sentinel 20 m)
import glob, rasterio
def grid(p):
    with rasterio.open(p) as ds:
        return ds.crs.to_string(), tuple(round(v, 3) for v in ds.transform[:6]), (ds.width, ds.height)
checks = {'Chuncheon_labels_AE_10m.tif': sorted(glob.glob('/content/drive/MyDrive/GEE_Export/Chuncheon_AE_2017_64Bands_int8*.tif')),
          'Chuncheon_labels_S2_20m.tif': sorted(glob.glob('/content/drive/MyDrive/GEE_Export/Chuncheon_S2_20*Bands*.tif'))}
for lab, refs in checks.items():
    g = grid(f'/content/data/{lab}')
    print(f'{lab}: {g}')
    for r in refs:
        gr = grid(r)
        print(f'   {"일치" if gr == g else "불일치"}  {r.split("/")[-1]}: {gr}')

Chuncheon_labels_AE_10m.tif: ('EPSG:32652', (10.0, 0.0, 368340.0, 0.0, -10.0, 4216220.0), (4648, 4550))
   불일치  Chuncheon_AE_2017_64Bands_int8.tif: ('EPSG:32652', (10.0, 0.0, 368350.0, 0.0, -10.0, 4216210.0), (4647, 4549))
Chuncheon_labels_S2_20m.tif: ('EPSG:32652', (20.0, 0.0, 368340.0, 0.0, -20.0, 4216220.0), (2324, 2275))
   일치  Chuncheon_S2_2019_2025_42Bands.tif: ('EPSG:32652', (20.0, 0.0, 368340.0, 0.0, -20.0, 4216220.0), (2324, 2275))


In [43]:
# [08-3] AlphaEarth 라벨을 실제 AlphaEarth 격자에 맞춤 (원점 1화소 차이 → 재표본 없이 잘라내기)
import numpy as np, rasterio, shutil
from rasterio.windows import from_bounds, Window
ref = sorted(glob.glob('/content/drive/MyDrive/GEE_Export/Chuncheon_AE_2017_64Bands_int8*.tif'))[0]
src_path = '/content/data/Chuncheon_labels_AE_10m.tif'
with rasterio.open(ref) as r, rasterio.open(src_path) as s:
    col = (r.transform.c - s.transform.c) / s.res[0]
    row = (s.transform.f - r.transform.f) / s.res[1]
    assert float(col).is_integer() and float(row).is_integer() and r.res == s.res and r.crs == s.crs, '격자 간격이 다름'
    win = Window(int(col), int(row), r.width, r.height)
    data = s.read(window=win, boundless=True, fill_value=255)
    prof = s.profile.copy(); prof.update(width=r.width, height=r.height, transform=r.transform)
    desc, tags = s.descriptions, s.tags()
tmp = '/content/data/_ae_labels.tif'
with rasterio.open(tmp, 'w', **prof) as d:
    d.write(data); d.descriptions = desc; d.update_tags(**tags)
shutil.move(tmp, src_path)
shutil.copy(src_path, '/content/drive/MyDrive/GEE_Export/Chuncheon_labels_AE_10m.tif')
print(f'잘라낸 위치: 열 {int(col)}, 행 {int(row)} →', grid(src_path), '| AlphaEarth:', grid(ref), '| 일치' if grid(src_path) == grid(ref) else '| 불일치')

잘라낸 위치: 열 1, 행 1 → ('EPSG:32652', (10.0, 0.0, 368350.0, 0.0, -10.0, 4216210.0), (4647, 4549)) | AlphaEarth: ('EPSG:32652', (10.0, 0.0, 368350.0, 0.0, -10.0, 4216210.0), (4647, 4549)) | 일치


## 비교 실험: Sentinel-2 S2-TSFM(2017~2025, 54밴드) vs AlphaEarth(2017~2025, 64차원)
같은 라벨(경영실적·임상도 합의, P2 = 2019→2025), 같은 공간 홀드아웃 블록, 같은 분류기(HistGradientBoosting)
1. 데이터 준비 → 2. S2-TSFM 54밴드 학습·평가 → 3. AlphaEarth 분류기 학습·평가 → 4. 비교표

In [45]:
# [08-4] 데이터 준비: 54밴드 Sentinel + AlphaEarth 9개 연도를 /content/data 로 복사, 격자 확인
import os, glob, shutil, rasterio
D = '/content/drive/MyDrive/GEE_Export'
os.makedirs('/content/data/ae', exist_ok=True)
for f in glob.glob(f'{D}/Chuncheon_S2_2017_2025_54Bands*.tif'):
    if not os.path.exists(f'/content/data/{os.path.basename(f)}'):
        shutil.copy(f, '/content/data/')
for f in sorted(glob.glob(f'{D}/Chuncheon_AE_20??_64Bands_int8*.tif')):
    if not os.path.exists(f'/content/data/ae/{os.path.basename(f)}'):
        shutil.copy(f, '/content/data/ae/')
def grid(p):
    with rasterio.open(p) as ds:
        return ds.crs.to_string(), tuple(round(v, 3) for v in ds.transform[:6]), (ds.width, ds.height)
s2 = sorted(glob.glob('/content/data/Chuncheon_S2_2017_2025_54Bands*.tif'))
print('S2 54밴드:', [os.path.basename(p) for p in s2])
print('  격자', grid(s2[0]), '| 20 m 라벨과', '일치' if grid(s2[0]) == grid('/content/data/Chuncheon_labels_S2_20m.tif') else '불일치')
ae = sorted(glob.glob('/content/data/ae/*.tif'))
print('AlphaEarth:', len(ae), '개', [os.path.basename(p)[13:17] for p in ae])
print('  격자 일치(라벨·연도 간):', all(grid(p) == grid('/content/data/Chuncheon_labels_AE_10m.tif') for p in ae))
!df -h /content | tail -1; free -g | head -2

S2 54밴드: ['Chuncheon_S2_2017_2025_54Bands.tif']
  격자 ('EPSG:32652', (20.0, 0.0, 368340.0, 0.0, -20.0, 4216220.0), (2324, 2275)) | 20 m 라벨과 일치
AlphaEarth: 9 개 ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
  격자 일치(라벨·연도 간): True
overlay         113G   61G   52G  55% /
               total        used        free      shared  buff/cache   available
Mem:              12           4           0           0           8           8


In [46]:
# [08-5] S2-TSFM: Sentinel-2 2017~2025 54밴드, P2(2019→2025) 합의 라벨 → 사전학습 30 epoch + 분류기 + 평가
s2_54 = sorted(glob.glob('/content/data/Chuncheon_S2_2017_2025_54Bands*.tif'))[0]
%run step05_s2_tsfm.py all --input {s2_54} --labels /content/data/Chuncheon_labels_S2_20m.tif --period P2 --outdir /content/runs/s2_54_p2 --epochs 30 --steps 100
shutil.copytree('/content/runs/s2_54_p2', f'{D}/compare_runs/s2_54_p2', dirs_exist_ok=True)

[입력] Chuncheon_S2_2017_2025_54Bands.tif: 9년 [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 2324×2275 px, 유효 52.9%, device cuda
[라벨] 새 형식 P2 (TRAIN·YEAR·CONS·SOURCE): no_change 1,025,990, harvest 17,635, conversion 721, gain 5,041 (국유림 검증 363,704 / 국유림 밖 685,683)
[분할] 홀드아웃 블록 화소 16.8% (블록 512px)
[pretrain] epoch 001/30 masked MAE train 0.2020 val 0.1290 (25s) *saved
[pretrain] epoch 002/30 masked MAE train 0.1193 val 0.1197 (26s) *saved
[pretrain] epoch 003/30 masked MAE train 0.1088 val 0.1089 (25s) *saved
[pretrain] epoch 004/30 masked MAE train 0.1021 val 0.1009 (25s) *saved
[pretrain] epoch 005/30 masked MAE train 0.1003 val 0.1054 (25s)
[pretrain] epoch 006/30 masked MAE train 0.0937 val 0.0981 (25s) *saved
[pretrain] epoch 007/30 masked MAE train 0.0957 val 0.0981 (25s) *saved
[pretrain] epoch 008/30 masked MAE train 0.0889 val 0.0976 (25s) *saved
[pretrain] epoch 009/30 masked MAE train 0.0917 val 0.0947 (25s) *saved
[pretrain] epoch 010/30 masked MAE train 0.0865 val 0.

'/content/drive/MyDrive/GEE_Export/compare_runs/s2_54_p2'

In [ ]:
%%writefile step08_ae_change_classifier.py
#!/usr/bin/env python3
# coding: utf-8
# [colab 정리본] 원본: s2_tsfm/ae_change_classifier.py — 파일명·모듈명만 실험 단계 번호에 맞춤 (계산 로직 동일)
"""AlphaEarth Foundations 연간 임베딩(2017~2025, 64차원, 10 m) 산림변화 분류기 — S2-TSFM 과 같은 조건으로 비교.

비교 조건 (step05_s2_tsfm.py 와 동일)
  - 라벨: build_training_label_rasters.py 의 TRAIN_P1/P2 (경영실적·임상도 합의), AlphaEarth 격자(10 m) 판
  - 공간 홀드아웃: Sentinel 20 m 격자의 512 px 블록(seed 42)을 그대로 10 m 화소에 대응 → 지리적으로 같은 검증 구역
  - 분류기·지표: step05_s2_tsfm.train_classifier / auc_ap (HistGradientBoosting, 클래스당 최대 6만 화소)
특징 (구간 y0→y1: P2 2019→2025, P1 2017→2019 — AlphaEarth 는 2017 부터)
  AE: e_y0, e_y1, e_y1−e_y0 (각 64), 연속 연도 코사인 거리 8, y0→y1 거리, 구간 내 최대 연속 거리
  AE+S2: 위 + Sentinel-2 지수 특징(step05_s2_tsfm.features 의 '지수만' 세트, 20 m 화소 값을 10 m 에 대응)
메모리: 연도별 1.35 GB(int8)를 한 해씩 읽어 라벨 화소만 뽑고, 지도는 행 띠 단위로 예측

Colab:
  %run step08_ae_change_classifier.py --ae-dir /content/data/ae --labels /content/data/Chuncheon_labels_AE_10m.tif \
       --s2 /content/data/Chuncheon_S2_2017_2025_54Bands.tif --period P2 --outdir /content/runs/ae_p2
"""
from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import Window

from step05_s2_tsfm import (FC_NAMES, auc_ap, block_split, load_stack, normalize, read_labels,
                         train_classifier, write_tif)

AE_YEARS = list(range(2017, 2026))
AE_NODATA = -128
PERIOD_YEARS = {"P2": (2019, 2025), "P1": (2017, 2019)}


def ae_path(ae_dir, year):
    hits = sorted(Path(ae_dir).glob(f"Chuncheon_AE_{year}_64Bands_int8*.tif"))
    if len(hits) != 1:
        raise FileNotFoundError(f"{ae_dir} 에서 {year} AlphaEarth 파일을 하나로 찾지 못함: {hits}")
    return hits[0]


def s2_index(ae_tf, s2_tf, rows, cols):
    """AlphaEarth 화소 중심 → Sentinel 20 m 화소 (행, 열)."""
    x = ae_tf.c + (cols + 0.5) * ae_tf.a
    y = ae_tf.f + (rows + 0.5) * ae_tf.e
    return ((y - s2_tf.f) / s2_tf.e).astype(int), ((x - s2_tf.c) / s2_tf.a).astype(int)


def unit(e):
    e = e.astype(np.float32) / 127.0
    return e / np.maximum(np.linalg.norm(e, axis=-1, keepdims=True), 1e-6)


def ae_features(embs, y0, y1):
    """embs: {year: [N, 64] int8} → 특징 [N, F], 이름, 비지도 점수 dict."""
    u = {y: unit(e) for y, e in embs.items()}
    cos = lambda a, b: 1.0 - (u[a] * u[b]).sum(-1)
    consec = np.stack([cos(a, b) for a, b in zip(AE_YEARS, AE_YEARS[1:])], 1)
    in_p = [i for i, (a, b) in enumerate(zip(AE_YEARS, AE_YEARS[1:])) if y0 <= a and b <= y1]
    d01 = cos(y0, y1)
    mx = consec[:, in_p].max(1)
    feats = np.concatenate([u[y0], u[y1], u[y1] - u[y0], consec, d01[:, None], mx[:, None]], 1)
    names = ([f"ae{y0}_{i:02d}" for i in range(64)] + [f"ae{y1}_{i:02d}" for i in range(64)]
             + [f"aedelta_{i:02d}" for i in range(64)] + [f"aecos_{a}_{b}" for a, b in zip(AE_YEARS, AE_YEARS[1:])]
             + [f"aecos_{y0}_{y1}", f"aecos_max_{y0}_{y1}"])
    year_max = np.asarray([AE_YEARS[i + 1] for i in in_p])[consec[:, in_p].argmax(1)]
    valid = np.all([((e != AE_NODATA) & (e != 0)).any(-1) for e in embs.values()], 0)  # 경계 밖은 0벡터로 저장됨
    return feats.astype(np.float32), names, {"ae_first_last": d01, "ae_max_consec": mx, "ae_max_year": year_max}, valid


def s2_features(s2_path, r2, c2):
    """step05_s2_tsfm.features 의 '지수만' 세트를 지정 화소에서 계산."""
    arr, valid, n, meta = load_stack(s2_path)
    stats = {"slope_mean": float(np.nanmean(arr[3 * n][valid])), "slope_std": float(max(1, np.nanstd(arr[3 * n][valid]))),
             "elev_mean": float(np.nanmean(arr[3 * n + 1][valid])), "elev_std": float(max(1, np.nanstd(arr[3 * n + 1][valid])))}
    x = normalize(arr, valid, n, stats)[:, r2, c2]
    del arr
    ndvi, ndwi, ndmi = x[0:3 * n:3] / 2, x[1:3 * n:3] / 2, x[2:3 * n:3] / 2
    f = np.concatenate([ndvi, ndwi, ndmi, np.diff(ndvi, axis=0), np.diff(ndmi, axis=0), x[3 * n:3 * n + 2],
                        np.stack([ndvi[-1] - ndvi[0], ndmi[-1] - ndmi[0], -np.diff(ndvi, axis=0).min(0)])]).T
    yrs = meta["years"]
    names = ([f"ndvi{y}" for y in yrs] + [f"ndwi{y}" for y in yrs] + [f"ndmi{y}" for y in yrs]
             + [f"dndvi{y}" for y in yrs[1:]] + [f"dndmi{y}" for y in yrs[1:]] + ["slope", "elev", "dNDVI", "dNDMI", "ndvi_max_drop"])
    return f.astype(np.float32), names, valid[r2, c2]


def evaluate_ae_scores(scores, labels, hold):
    fc, mg, nat = labels["FC_TYPE"], labels["MGMT"], labels["SOURCE"] == 1
    tasks = {
        "임상도 변화여부(벌채·전용·증가 vs 무변화)": (np.isin(fc, [1, 2, 3]), fc == 0),
        "국유림 검증라벨 변화여부(실적·임상도 합의)": (np.isin(fc, [1, 2, 3]) & nat, (fc == 0) & nat),
        "임상도 벌채·갱신 vs 무변화": (fc == 1, fc == 0),
        "국유림 실적 수확·갱신 vs 실적없음": (mg == 1, mg == 0),
        "국유림 실적 솎아베기 vs 실적없음": (mg == 2, mg == 0),
    }
    rows = []
    for task, (pos, neg) in tasks.items():
        m = hold & (pos | neg)
        y = pos[m].astype(int)
        for name, key in (("AE 구간 첫해→끝해 코사인거리", "ae_first_last"), ("AE 구간 최대 연속 코사인거리", "ae_max_consec")):
            auc, ap = auc_ap(y, scores[key][m])
            rows.append({"task": task, "score": name, "n_pos": int(y.sum()), "n_neg": int((1 - y).sum()),
                         "auroc": round(auc, 4), "ap": round(ap, 4)})
    return rows


def predict_map(clf, ae_dir, y0, y1, profile, outdir, strip=128):  # 128행 띠 ≈ 9년×64밴드 int8 340 MB
    files = {y: rasterio.open(ae_path(ae_dir, y)) for y in AE_YEARS}
    h, w = profile["height"], profile["width"]
    cls_map = np.full((h, w), 255, np.uint8)
    prob = np.full((len(clf.classes_), h, w), np.nan, np.float32)
    try:
        for r in range(0, h, strip):
            win = Window(0, r, w, min(strip, h - r))
            embs = {y: f.read(window=win).reshape(64, -1).T for y, f in files.items()}
            F, _, _, valid = ae_features(embs, y0, y1)
            if valid.any():
                p = clf.predict_proba(F[valid])
                rows = slice(r, r + win.height)
                flat_c = cls_map[rows].reshape(-1)
                flat_c[valid] = clf.classes_[p.argmax(1)]
                cls_map[rows] = flat_c.reshape(win.height, w)
                for k in range(len(clf.classes_)):
                    flat_p = prob[k, rows].reshape(-1)
                    flat_p[valid] = p[:, k]
                    prob[k, rows] = flat_p.reshape(win.height, w)
            if (r // strip) % 8 == 7:
                print(f"  [map] 행 {r + win.height}/{h}", flush=True)
    finally:
        for f in files.values():
            f.close()
    write_tif(outdir / "ae_change_class.tif", profile, {"change_class": cls_map}, dtype="uint8", nodata=255)
    write_tif(outdir / "ae_change_prob.tif", profile, {f"p_{FC_NAMES[k]}": prob[i] for i, k in enumerate(clf.classes_)})
    return cls_map


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ae-dir", required=True)
    p.add_argument("--labels", required=True, help="AlphaEarth 격자(10 m) 학습 라벨 래스터")
    p.add_argument("--s2", default=None, help="Sentinel 지수 스택 (AE+S2 결합 특징·홀드아웃 블록 격자)")
    p.add_argument("--s2-labels", default=None, help="Sentinel 20 m 라벨(--s2 없을 때 블록 격자 기준)")
    p.add_argument("--period", choices=["P1", "P2"], default="P2")
    p.add_argument("--outdir", default="runs/ae")
    p.add_argument("--block", type=int, default=512, help="Sentinel 20 m 기준 홀드아웃 블록(px) — step05_s2_tsfm 와 같아야 함")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--no-map", action="store_true")
    args, _ = p.parse_known_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    y0, y1 = PERIOD_YEARS[args.period]

    labels_full, (w, h), fmt = read_labels(args.labels, args.period)
    with rasterio.open(args.labels) as lb:
        ae_tf, profile = lb.transform, lb.profile.copy()
    with rasterio.open(ae_path(args.ae_dir, AE_YEARS[0])) as ref:
        if (ref.width, ref.height, ref.transform) != (w, h, ae_tf):
            raise ValueError("라벨 격자가 AlphaEarth 격자와 다름 — build_training_label_rasters.py --ae-ref 로 다시 만드세요")
        profile.update(crs=ref.crs)
    grid_src = args.s2 or args.s2_labels
    with rasterio.open(grid_src) as s:
        s2_tf, s2_h, s2_w = s.transform, s.height, s.width
    hold_s2 = block_split(s2_h, s2_w, args.block, args.seed)

    # 평가·학습에 쓸 화소: 홀드아웃 블록의 모든 라벨 화소 + 학습 블록의 클래스당 최대 6만 화소
    fc, mg = labels_full["FC_TYPE"].ravel(), labels_full["MGMT"].ravel()
    rows_all, cols_all = np.divmod(np.arange(h * w), w)
    r2, c2 = s2_index(ae_tf, s2_tf, rows_all, cols_all)
    inside = (r2 >= 0) & (r2 < s2_h) & (c2 >= 0) & (c2 < s2_w)
    hold = np.zeros(h * w, bool)
    hold[inside] = hold_s2[r2[inside], c2[inside]]
    rng = np.random.default_rng(args.seed)
    test_idx = np.where(hold & inside & ((fc != 255) | (mg != 255)))[0]
    train_idx = np.concatenate([rng.permutation(np.where(~hold & inside & (fc == k))[0])[:60000] for k in FC_NAMES])
    idx = np.sort(np.concatenate([train_idx, test_idx]))
    del rows_all, cols_all
    print(f"[라벨] {fmt}, 격자 {w}×{h} 10 m, 구간 {y0}→{y1}")
    print(f"[표본] 학습 {len(train_idx):,} 화소 (클래스당 ≤6만) / 홀드아웃 평가 {len(test_idx):,} 화소")

    embs = {}
    for y in AE_YEARS:
        t = time.time()
        with rasterio.open(ae_path(args.ae_dir, y)) as ds:
            embs[y] = ds.read().reshape(64, -1)[:, idx].T.copy()
        print(f"[AE] {y} 읽기 {time.time() - t:.0f}s", flush=True)
    F_ae, n_ae, scores, valid = ae_features(embs, y0, y1)
    del embs
    lab = {k: v.ravel()[idx] for k, v in labels_full.items()}
    hold_s = hold[idx]
    lab["FC_TYPE"] = np.where(valid, lab["FC_TYPE"], 255)

    score_rows = evaluate_ae_scores(scores, lab, hold_s & valid)
    results, clf_ae = {}, None
    sets = [("AE", F_ae, n_ae)]
    if args.s2:
        F_s2, n_s2, s2_valid = s2_features(args.s2, r2[idx], c2[idx])
        lab["FC_TYPE"] = np.where(s2_valid, lab["FC_TYPE"], 255)
        sets.append(("AE+S2지수", np.concatenate([F_ae, F_s2], 1), n_ae + n_s2))
        sets.append(("S2지수(10m 대응)", F_s2, n_s2))
    for tag, F, names in sets:
        clf, met = train_classifier(F, names, lab, hold_s, args.seed)
        met["n_features"] = len(names)
        results[tag] = met
        print(f"[classifier] {tag}: 특징 {len(names)}개, balanced acc {met['balanced_accuracy']}, macro F1 {met['macro_f1']}, "
              f"변화 AUROC {met['changed_auroc']}, AP {met['changed_ap']} | 국유림 검증 AUROC {met['verified_changed_auroc']}, "
              f"AP {met['verified_changed_ap']}", flush=True)
        if tag == "AE":
            clf_ae = clf

    fy = lab["FC_YEAR"].astype(float)
    fy[fy == 255] = np.nan
    m = hold_s & np.isin(lab["FC_TYPE"], [1, 2, 3]) & np.isfinite(fy) & (fy + 2000 >= y0)
    diff = np.abs(scores["ae_max_year"][m] - (fy[m] + 2000))
    year_acc = {"n": int(m.sum()), "exact": round(float((diff == 0).mean()), 4) if m.any() else None,
                "within_1yr": round(float((diff <= 1).mean()), 4) if m.any() else None}

    class_area = None
    if not args.no_map:
        print("[map] 전체 영상 AE 분류 지도 예측 …", flush=True)
        cls_map = predict_map(clf_ae, args.ae_dir, y0, y1, profile, outdir)
        class_area = {FC_NAMES[k]: round(float((cls_map == k).sum()) * 0.01, 1) for k in FC_NAMES}
        import joblib
        joblib.dump({"clf": clf_ae, "features": n_ae, "classes": FC_NAMES, "period": args.period}, outdir / "step08_ae_change_classifier.joblib")

    report = {"model": "AlphaEarth V1 annual embedding", "period": args.period, "years": [y0, y1],
              "unsupervised_scores": score_rows, "classifier": results, "change_year": year_acc,
              "class_area_ha": class_area, "elapsed_min": round((time.time() - t0) / 60, 1)}
    (outdir / "report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    print("\n[AE 비지도 점수 AUROC / AP — 홀드아웃 블록]")
    for r in score_rows:
        print(f"  {r['task']:<32} {r['score']:<24} AUROC {r['auroc']:.3f}  AP {r['ap']:.3f}  (양성 {r['n_pos']:,} / 음성 {r['n_neg']:,})")
    print("[변화 연도 ±1년]", year_acc)
    print("[분류 면적 ha]", class_area)
    print(f"[완료] {report['elapsed_min']}분, 결과: {outdir}")


if __name__ == "__main__":
    main()


In [48]:
# [08-6] AlphaEarth 2017~2025 분류기 (P2 2019→2025): AE / AE+S2지수 / S2지수(같은 10m 화소) 비교 + AE 분류 지도
%run step08_ae_change_classifier.py --ae-dir /content/data/ae --labels /content/data/Chuncheon_labels_AE_10m.tif --s2 {s2_54} --period P2 --outdir /content/runs/ae_p2
shutil.copytree('/content/runs/ae_p2', f'{D}/compare_runs/ae_p2', dirs_exist_ok=True)

[라벨] 새 형식 P2 (TRAIN·YEAR·CONS·SOURCE), 격자 4647×4549 10 m, 구간 2019→2025
[표본] 학습 136,462 화소 (클래스당 ≤6만) / 홀드아웃 평가 774,165 화소
[AE] 2017 읽기 11s
[AE] 2018 읽기 11s
[AE] 2019 읽기 11s
[AE] 2020 읽기 12s
[AE] 2021 읽기 11s
[AE] 2022 읽기 11s
[AE] 2023 읽기 11s
[AE] 2024 읽기 12s
[AE] 2025 읽기 11s
[classifier] AE: 특징 202개, balanced acc 0.6932, macro F1 0.5781, 변화 AUROC 0.9909, AP 0.8714 | 국유림 검증 AUROC 0.9996, AP 0.9927
[classifier] AE+S2지수: 특징 250개, balanced acc 0.7201, macro F1 0.5853, 변화 AUROC 0.9918, AP 0.8792 | 국유림 검증 AUROC 0.9995, AP 0.9934
[classifier] S2지수(10m 대응): 특징 48개, balanced acc 0.6832, macro F1 0.518, 변화 AUROC 0.9757, AP 0.8197 | 국유림 검증 AUROC 0.997, AP 0.9758
[map] 전체 영상 AE 분류 지도 예측 …
  [map] 행 1024/4549
  [map] 행 2048/4549
  [map] 행 3072/4549
  [map] 행 4096/4549

[AE 비지도 점수 AUROC / AP — 홀드아웃 블록]
  임상도 변화여부(벌채·전용·증가 vs 무변화)        AE 구간 첫해→끝해 코사인거리        AUROC 0.961  AP 0.709  (양성 16,409 / 음성 634,718)
  임상도 변화여부(벌채·전용·증가 vs 무변화)        AE 구간 최대 연속 코사인거리        AUROC 0.950  AP 0.712  (양성 16,409

'/content/drive/MyDrive/GEE_Export/compare_runs/ae_p2'

In [49]:
# [08-7] 비교표: Sentinel-2 S2-TSFM(54밴드) vs AlphaEarth — 같은 라벨(P2 2019→2025 합의)·같은 공간 홀드아웃 블록
import os, json, pandas as pd
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
D = '/content/drive/MyDrive/GEE_Export/compare_runs'
def load(name):
    for p in (f'/content/runs/{name}/report.json', f'{D}/{name}/report.json'):
        if os.path.exists(p):
            print('읽음:', p)
            return json.load(open(p, encoding='utf-8'))
    raise FileNotFoundError(name)
s2r, aer = load('s2_54_p2'), load('ae_p2')

rows = []
for src, rep, res in (('Sentinel-2 S2-TSFM', s2r, '20m'), ('AlphaEarth', aer, '10m')):
    for tag, m in rep['classifier'].items():
        rows.append({'자료': src, '해상도': res, '특징': tag, '특징수': m['n_features'],
                     'balanced_acc': m['balanced_accuracy'], 'macro_F1': m['macro_f1'],
                     '변화_AUROC': m['changed_auroc'], '변화_AP': m['changed_ap'],
                     '국유림검증_AUROC': m['verified_changed_auroc'], '국유림검증_AP': m['verified_changed_ap'],
                     **{f'F1_{k}': v['f1-score'] for k, v in m['per_class'].items()},
                     **{f'재현율_{k}': v['recall'] for k, v in m['per_class'].items()}, '평가화소': m['n_test']})
clf_tbl = pd.DataFrame(rows)
uns = pd.DataFrame([{**r, '자료': 'Sentinel-2'} for r in s2r['unsupervised_scores']] +
                   [{**r, '자료': 'AlphaEarth'} for r in aer['unsupervised_scores']])
best = uns.loc[uns.groupby(['task', '자료'])['auroc'].idxmax()]
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print('\n[분류기 — 홀드아웃 블록]')
print(clf_tbl.round(4).to_string(index=False))
print('\n[비지도 변화점수 — 과제별 최고 AUROC (점수 이름)]')
print(best[['task', '자료', 'score', 'auroc', 'ap', 'n_pos', 'n_neg']].to_string(index=False))
print('\n[변화 연도] S2:', s2r['change_year'], '\n           AE:', aer['change_year'])
print('[분류 면적 ha] S2(20m):', s2r['class_area_ha'], '\n               AE(10m):', aer['class_area_ha'])
print('[실행시간 분] S2:', s2r['elapsed_min'], '| AE:', aer['elapsed_min'])
clf_tbl.to_csv(f'{D}/비교_분류기.csv', index=False, encoding='utf-8-sig')
uns.to_csv(f'{D}/비교_비지도점수.csv', index=False, encoding='utf-8-sig')
print('\n저장:', D, '| 산출물:', sorted(os.listdir(f'{D}/ae_p2')))

읽음: /content/runs/s2_54_p2/report.json
읽음: /content/runs/ae_p2/report.json

[분류기 — 홀드아웃 블록]
                 자료 해상도           특징  특징수  balanced_acc  macro_F1  변화_AUROC  변화_AP  국유림검증_AUROC  국유림검증_AP  F1_no_change  F1_harvest  F1_conversion  F1_gain  재현율_no_change  재현율_harvest  재현율_conversion  재현율_gain   평가화소
Sentinel-2 Mini-GFM 20m          지수만   48        0.6711    0.5020    0.9789 0.8215       0.9980    0.9731        0.9814      0.6625         0.0676   0.2963         0.9666       0.9411          0.3037    0.4729 162820
Sentinel-2 Mini-GFM 20m       지수+GFM   82        0.6768    0.5142    0.9808 0.8219       0.9979    0.9752        0.9833      0.6656         0.0971   0.3106         0.9706       0.9516          0.3407    0.4444 162820
         AlphaEarth 10m           AE  202        0.6932    0.5781    0.9909 0.8714       0.9996    0.9927        0.9914      0.8172         0.1662   0.3378         0.9866       0.9629          0.4732    0.3500 651127
         AlphaEarth 10m      AE+S2지수  25

In [1]:
# [08-8] 진단: AlphaEarth 분류 지도 면적 이상 (춘천 밖 화소가 '유효'로 분류됐는지) — 메모리 절약을 위해 띠 단위
import numpy as np, rasterio, glob
from rasterio.windows import Window
ae25 = sorted(glob.glob('/content/data/ae/Chuncheon_AE_2025_64Bands_int8*.tif'))[0]
cnt = {}
with rasterio.open(ae25) as src, rasterio.open('/content/runs/ae_p2/ae_change_class.tif') as cl:
    print('nodata 태그:', src.nodata)
    for r in range(0, src.height, 512):
        win = Window(0, r, src.width, min(512, src.height - r))
        e = src.read(window=win)
        c = cl.read(1, window=win)
        kind = np.where((e == -128).all(0), '-128', np.where((e == 0).all(0), '0벡터', '정상'))
        for k in ('-128', '0벡터', '정상'):
            v, n = np.unique(c[kind == k], return_counts=True)
            for vv, nn in zip(v, n):
                cnt[(k, int(vv))] = cnt.get((k, int(vv)), 0) + int(nn)
for k in ('-128', '0벡터', '정상'):
    d = {v: n for (kk, v), n in cnt.items() if kk == k}
    print(f'{k} 화소 {sum(d.values()):,} → 분류 {d}  (ha: { {v: round(n * 0.01) for v, n in d.items()} })')

nodata 태그: -128.0
-128 화소 0 → 분류 {}  (ha: {})
0벡터 화소 9,972,054 → 분류 {1: 9972054}  (ha: {1: 99721})
정상 화소 11,167,149 → 분류 {0: 9983280, 1: 466405, 2: 456900, 3: 260564}  (ha: {0: 99833, 1: 4664, 2: 4569, 3: 2606})


In [2]:
# [08-9] 수정: 춘천 밖 0벡터 화소를 AE 분류·확률 지도에서 제외(255/NaN) → 면적 재계산, report.json·Drive 갱신
import json, shutil, numpy as np, rasterio
from rasterio.windows import Window
run = '/content/runs/ae_p2'
counts = np.zeros(4, np.int64)
with rasterio.open(ae25) as src, rasterio.open(f'{run}/ae_change_class.tif', 'r+') as cl, \
        rasterio.open(f'{run}/ae_change_prob.tif', 'r+') as pr:
    for r in range(0, src.height, 512):
        win = Window(0, r, src.width, min(512, src.height - r))
        zero = (src.read(window=win) == 0).all(0)
        c = cl.read(1, window=win); c[zero] = 255; cl.write(c, 1, window=win)
        p = pr.read(window=win); p[:, zero] = pr.nodata; pr.write(p, window=win)
        counts += np.bincount(c[c != 255].ravel(), minlength=4)[:4]
rep = json.load(open(f'{run}/report.json', encoding='utf-8'))
rep['class_area_ha'] = {k: round(float(n) * 0.01, 1) for k, n in zip(['no_change', 'harvest', 'conversion', 'gain'], counts)}
rep['note'] = '춘천 경계 밖 0벡터 화소 제외 후 면적 (분류기 지표는 경계 안 라벨 화소만 사용해 영향 없음)'
json.dump(rep, open(f'{run}/report.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
shutil.copytree(run, '/content/drive/MyDrive/GEE_Export/compare_runs/ae_p2', dirs_exist_ok=True)
print('AE 분류 면적 ha (수정):', rep['class_area_ha'], '| 합계', round(sum(rep['class_area_ha'].values())))
!sed -i 's/valid = np.all(\[(e != AE_NODATA).any(-1) for e in embs.values()\], 0)/valid = np.all([((e != AE_NODATA) \& (e != 0)).any(-1) for e in embs.values()], 0)  # 경계 밖은 0벡터로 저장됨/' step08_ae_change_classifier.py
!grep -n "valid = np.all" step08_ae_change_classifier.py

AE 분류 면적 ha (수정): {'no_change': 99832.8, 'harvest': 4664.1, 'conversion': 4569.0, 'gain': 2605.6} | 합계 111672
70:    valid = np.all([((e != AE_NODATA) & (e != 0)).any(-1) for e in embs.values()], 0)  # 경계 밖은 0벡터로 저장됨


In [3]:
# [08-10] 산림 마스크 업로드: C:\src\ForestChangeGeoAI\output 의
#   Chuncheon_forestmask_S2_20m.tif + Chuncheon_forestmask_AE_10m.tif 두 파일을 함께 선택 → /content/data + Drive 저장
import os, shutil
from google.colab import files
names = ['Chuncheon_forestmask_S2_20m.tif', 'Chuncheon_forestmask_AE_10m.tif']
G = '/content/drive/MyDrive/GEE_Export'
for n in names:
    if not os.path.exists(f'/content/data/{n}') and os.path.exists(f'{G}/{n}'):
        shutil.copy(f'{G}/{n}', f'/content/data/{n}')
if not all(os.path.exists(f'/content/data/{n}') for n in names):
    for fn in files.upload():
        shutil.move(fn, f'/content/data/{fn}')
        shutil.copy(f'/content/data/{fn}', f'{G}/{fn}')
print({n: os.path.exists(f'/content/data/{n}') for n in names})

Saving Chuncheon_forestmask_AE_10m.tif to Chuncheon_forestmask_AE_10m.tif
Saving Chuncheon_forestmask_S2_20m.tif to Chuncheon_forestmask_S2_20m.tif
{'Chuncheon_forestmask_S2_20m.tif': True, 'Chuncheon_forestmask_AE_10m.tif': True}


In [4]:
# [08-11] 산림 마스크 적용: 두 분류 지도의 비산림 화소를 255(제외)로 → 면적·평가 재계산, 임상도 라벨 면적과 비교
import json, shutil, numpy as np, rasterio, pandas as pd
from rasterio.windows import Window
NAMES = ['no_change', 'harvest', 'conversion', 'gain']
runs = {'Sentinel-2 S2-TSFM (20m)': ('/content/runs/s2_54_p2/change_class.tif', '/content/data/Chuncheon_forestmask_S2_20m.tif',
                                      '/content/data/Chuncheon_labels_S2_20m.tif'),
        'AlphaEarth (10m)': ('/content/runs/ae_p2/ae_change_class.tif', '/content/data/Chuncheon_forestmask_AE_10m.tif',
                             '/content/data/Chuncheon_labels_AE_10m.tif')}
rows = []
for name, (cls_path, mask_path, lab_path) in runs.items():
    out_path = cls_path.replace('.tif', '_forest.tif')
    before, after = np.zeros(4, np.int64), np.zeros(4, np.int64)
    conf = np.zeros((4, 4), np.int64)  # 라벨(TRAIN_P2) × 마스크 후 예측, 전체 라벨 화소(학습+검증)
    with rasterio.open(cls_path) as src, rasterio.open(mask_path) as mk, rasterio.open(lab_path) as lb:
        assert (src.width, src.height, src.transform) == (mk.width, mk.height, mk.transform), '마스크 격자 불일치'
        prof = src.profile.copy()
        px_ha = abs(src.transform.a * src.transform.e) / 1e4
        tb = list(lb.descriptions).index('TRAIN_P2') + 1
        with rasterio.open(out_path, 'w', **prof) as dst:
            dst.set_band_description(1, 'change_class_forestmask')
            for r in range(0, src.height, 512):
                win = Window(0, r, src.width, min(512, src.height - r))
                c, m, t = src.read(1, window=win), mk.read(1, window=win), lb.read(tb, window=win)
                before += np.bincount(c[c != 255].ravel(), minlength=4)[:4]
                c = np.where(m == 1, c, 255).astype(np.uint8)
                after += np.bincount(c[c != 255].ravel(), minlength=4)[:4]
                ok = (c != 255) & (t != 255)
                np.add.at(conf, (t[ok], c[ok]), 1)
                dst.write(c, 1, window=win)
    lab_outside = None
    rows.append({'모델': name, **{f'{k}_전_ha': round(b * px_ha) for k, b in zip(NAMES, before)},
                 **{f'{k}_후_ha': round(a * px_ha) for k, a in zip(NAMES, after)},
                 **{f'{k}_라벨재현율': round(conf[i, i] / max(conf[i].sum(), 1), 3) for i, k in enumerate(NAMES)}})
    rep_path = cls_path.rsplit('/', 1)[0] + '/report.json'
    rep = json.load(open(rep_path, encoding='utf-8'))
    rep['class_area_ha_forestmask'] = {k: round(float(a * px_ha), 1) for k, a in zip(NAMES, after)}
    rep['forestmask'] = '임상도 2019 또는 2025 임지(입목지·무립목지) 화소만 — 두 시기 모두 비산림·도화범위 밖 제외'
    json.dump(rep, open(rep_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    run_dir = cls_path.rsplit('/', 1)[0]
    shutil.copytree(run_dir, f'/content/drive/MyDrive/GEE_Export/compare_runs/{run_dir.rsplit("/", 1)[1]}', dirs_exist_ok=True)
tbl = pd.DataFrame(rows).set_index('모델').T
print(tbl.to_string())
print('\n참고 — 임상도 2019→2025 라벨 면적(3시기 모두 도화 구역): 벌채·갱신 1,573 ha, 산지전용 256 ha, 산림증가 464 ha')
tbl.to_csv('/content/drive/MyDrive/GEE_Export/compare_runs/비교_산림마스크_면적.csv', encoding='utf-8-sig')

모델                Sentinel-2 Mini-GFM (20m)  AlphaEarth (10m)
no_change_전_ha                    84690.000         99833.000
harvest_전_ha                       5275.000          4664.000
conversion_전_ha                   14224.000          4569.000
gain_전_ha                          7621.000          2606.000
no_change_후_ha                    77906.000         79915.000
harvest_후_ha                       3007.000          2627.000
conversion_후_ha                    2126.000          1113.000
gain_후_ha                          1997.000          1385.000
no_change_라벨재현율                       0.972             0.988
harvest_라벨재현율                         0.976             0.994
conversion_라벨재현율                      0.856             0.896
gain_라벨재현율                            0.846             0.827

참고 — 임상도 2019→2025 라벨 면적(3시기 모두 도화 구역): 벌채·갱신 1,573 ha, 산지전용 256 ha, 산림증가 464 ha


In [5]:
# [08-12] 최소 면적 필터(체 거르기, 8방향 연결): 0.1 / 0.2 / 0.5 ha(임상도 최소 구획) → 면적·홀드아웃 정확도 비교
#      작은 조각은 경계를 가장 많이 맞댄 이웃 클래스로 병합 (변화 소조각 제거 + 변화지 안 작은 무변화 구멍 메움)
import json, math, shutil, numpy as np, rasterio, pandas as pd
from rasterio.features import sieve
from sklearn.metrics import precision_recall_fscore_support
from step05_s2_tsfm import block_split
NAMES = ['no_change', 'harvest', 'conversion', 'gain']
MMU_HA = [0.1, 0.2, 0.5]
with rasterio.open('/content/data/Chuncheon_labels_S2_20m.tif') as s2l:
    s2_tf, s2_h, s2_w = s2l.transform, s2l.height, s2l.width
hold_s2 = block_split(s2_h, s2_w, 512, 42)  # step05_s2_tsfm·step08_ae_change_classifier 와 같은 홀드아웃 블록

def holdout_mask(tf, h, w):
    if (tf, h, w) == (s2_tf, s2_h, s2_w):
        return hold_s2
    rows, cols = np.mgrid[0:h, 0:w]
    r2 = ((tf.f + (rows + 0.5) * tf.e - s2_tf.f) / s2_tf.e).astype(int)
    c2 = ((tf.c + (cols + 0.5) * tf.a - s2_tf.c) / s2_tf.a).astype(int)
    ok = (r2 >= 0) & (r2 < s2_h) & (c2 >= 0) & (c2 < s2_w)
    out = np.zeros((h, w), bool); out[ok] = hold_s2[r2[ok], c2[ok]]
    return out

def scores(cls, lab, hold):
    m = hold & (lab != 255) & (cls != 255)
    p, r, f, _ = precision_recall_fscore_support(lab[m], cls[m], labels=[0, 1, 2, 3], zero_division=0)
    return {'macro_F1': round(f.mean(), 4), **{f'F1_{k}': round(v, 3) for k, v in zip(NAMES, f)},
            **{f'정밀도_{k}': round(v, 3) for k, v in zip(NAMES[1:], p[1:])}, **{f'재현율_{k}': round(v, 3) for k, v in zip(NAMES[1:], r[1:])}}

runs = {'S2-TSFM 20m': ('/content/runs/s2_54_p2', 'change_class_forest.tif', '/content/data/Chuncheon_labels_S2_20m.tif'),
        'AlphaEarth 10m': ('/content/runs/ae_p2', 'ae_change_class_forest.tif', '/content/data/Chuncheon_labels_AE_10m.tif')}
rows = []
for name, (run, fn, lab_path) in runs.items():
    with rasterio.open(f'{run}/{fn}') as src, rasterio.open(lab_path) as lb:
        cls, prof = src.read(1), src.profile.copy()
        lab = lb.read(list(lb.descriptions).index('TRAIN_P2') + 1)
    px_ha = abs(prof['transform'].a * prof['transform'].e) / 1e4
    hold = holdout_mask(prof['transform'], prof['height'], prof['width'])
    valid = cls != 255
    area = lambda c: {f'{k}_ha': round(float((c == i).sum()) * px_ha) for i, k in enumerate(NAMES)}
    rows.append({'모델': name, 'MMU_ha': 0, '최소화소': 1, **area(cls), **scores(cls, lab, hold)})
    for mmu in MMU_HA:
        n_px = math.ceil(mmu / px_ha)
        sv = sieve(cls, size=n_px, mask=valid, connectivity=8).astype(np.uint8)
        sv[~valid] = 255
        rows.append({'모델': name, 'MMU_ha': mmu, '최소화소': n_px, **area(sv), **scores(sv, lab, hold)})
        out = f'{run}/{fn.replace(".tif", f"_mmu{str(mmu).replace(".", "")}ha.tif")}'
        with rasterio.open(out, 'w', **prof) as dst:
            dst.write(sv, 1); dst.set_band_description(1, f'change_class_forestmask_mmu{mmu}ha')
    shutil.copytree(run, f'/content/drive/MyDrive/GEE_Export/compare_runs/{run.rsplit("/", 1)[1]}', dirs_exist_ok=True)
tbl = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(tbl.to_string(index=False))
print('\n참고 — 임상도 2019→2025 라벨 면적: 벌채·갱신 1,573 ha, 산지전용 256 ha, 산림증가 464 ha')
tbl.to_csv('/content/drive/MyDrive/GEE_Export/compare_runs/비교_최소면적필터.csv', index=False, encoding='utf-8-sig')

            모델  MMU_ha  최소화소  no_change_ha  harvest_ha  conversion_ha  gain_ha  macro_F1  F1_no_change  F1_harvest  F1_conversion  F1_gain  정밀도_harvest  정밀도_conversion  정밀도_gain  재현율_harvest  재현율_conversion  재현율_gain
  Mini-GFM 20m     0.0     1         77906        3007           2126     1997    0.5142         0.983       0.666          0.097    0.311        0.512           0.057     0.239        0.952           0.341     0.444
  Mini-GFM 20m     0.1     3         79274        2642           1442     1679    0.5420         0.987       0.718          0.117    0.346        0.575           0.075     0.287        0.953           0.267     0.437
  Mini-GFM 20m     0.2     5         79833        2492           1144     1568    0.5569         0.989       0.747          0.130    0.361        0.615           0.089     0.309        0.953           0.237     0.434
  Mini-GFM 20m     0.5    13         80727        2232            714     1363    0.5705         0.992       0.803          0.103   

In [ ]:
%%writefile step08_final_change_map.py
#!/usr/bin/env python3
# coding: utf-8
# [colab 정리본] 원본: s2_tsfm/final_change_map.py — 파일명·모듈명만 실험 단계 번호에 맞춤 (계산 로직 동일)
"""춘천 최종 산림변화 지도 (2019→2025): AlphaEarth + Sentinel-2 지수 분류기, 산림 마스크, 클래스별 최소 면적.

1) 학습: step08_ae_change_classifier 와 같은 표본(합의 라벨 TRAIN_P2, 같은 공간 홀드아웃 블록)으로 AE+S2지수 분류기
2) 예측: 산림 마스크(임상도 2019·2025 임지) 안 10 m 화소만, 128행 띠 단위
3) 후처리(최소 면적, 8방향 연결 체 거르기): 벌채·갱신·산림증가 0.5 ha(임상도 최소 구획), 산지전용 0.2 ha
4) 산출 (outdir)
   final_change_class.tif  0 무변화, 1 벌채·갱신, 2 산지전용, 3 산림증가, 255 비산림·자료없음
   final_change_prob.tif   클래스별 확률 4밴드
   final_change_year.tif   변화 화소의 추정 발생연도(AlphaEarth 연속연도 코사인 거리 최대 연도)
   final_change_patches.gpkg  변화 패치 폴리곤 (클래스, 면적 ha, 평균 확률, 최빈 발생연도)
   final_report.json, final_preview.png

Colab:
  %run step08_final_change_map.py --ae-dir /content/data/ae --labels /content/data/Chuncheon_labels_AE_10m.tif \
       --s2 /content/data/Chuncheon_S2_2017_2025_54Bands.tif --forest-mask /content/data/Chuncheon_forestmask_AE_10m.tif \
       --outdir /content/runs/final_p2
"""
from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import rasterio
from rasterio.features import shapes, sieve
from rasterio.windows import Window
from scipy import ndimage

from step08_ae_change_classifier import AE_YEARS, PERIOD_YEARS, ae_features, ae_path, s2_features, s2_index
from step05_s2_tsfm import FC_NAMES, block_split, read_labels, train_classifier, write_tif

CLASS_KO = {0: "무변화", 1: "벌채·갱신", 2: "산지전용", 3: "산림증가"}


def samples(labels_full, w, h, ae_tf, s2_tf, s2_h, s2_w, hold_s2, seed):
    """step08_ae_change_classifier.main 과 같은 표본: 홀드아웃 블록 라벨 화소 전부 + 학습 블록 클래스당 ≤6만."""
    fc, mg = labels_full["FC_TYPE"].ravel(), labels_full["MGMT"].ravel()
    rows, cols = np.divmod(np.arange(h * w), w)
    r2, c2 = s2_index(ae_tf, s2_tf, rows, cols)
    inside = (r2 >= 0) & (r2 < s2_h) & (c2 >= 0) & (c2 < s2_w)
    hold = np.zeros(h * w, bool)
    hold[inside] = hold_s2[r2[inside], c2[inside]]
    rng = np.random.default_rng(seed)
    test_idx = np.where(hold & inside & ((fc != 255) | (mg != 255)))[0]
    train_idx = np.concatenate([rng.permutation(np.where(~hold & inside & (fc == k))[0])[:60000] for k in FC_NAMES])
    idx = np.sort(np.concatenate([train_idx, test_idx]))
    return idx, hold, r2, c2


def class_mmu(cls, valid, px_ha, mmu):
    """클래스별 최소 면적: 큰 기준(0.5 ha)으로 거른 지도에, 작은 기준 클래스(산지전용 0.2 ha)의 결과를 덮어씀."""
    out = None
    for thr in sorted(set(mmu.values()), reverse=True):
        sv = sieve(cls, size=math.ceil(thr / px_ha), mask=valid, connectivity=8).astype(np.uint8)
        keep = [k for k, v in mmu.items() if v == thr]
        out = sv if out is None else np.where(np.isin(sv, keep), sv, out)
    out[~valid] = 255
    return out.astype(np.uint8)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ae-dir", required=True)
    p.add_argument("--labels", required=True)
    p.add_argument("--s2", required=True)
    p.add_argument("--forest-mask", required=True)
    p.add_argument("--period", choices=["P1", "P2"], default="P2")
    p.add_argument("--outdir", default="runs/final")
    p.add_argument("--block", type=int, default=512)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--mmu-harvest", type=float, default=0.5)
    p.add_argument("--mmu-conversion", type=float, default=0.2)
    p.add_argument("--mmu-gain", type=float, default=0.5)
    p.add_argument("--strip", type=int, default=128)
    args, _ = p.parse_known_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    y0, y1 = PERIOD_YEARS[args.period]
    mmu = {1: args.mmu_harvest, 2: args.mmu_conversion, 3: args.mmu_gain}

    labels_full, (w, h), fmt = read_labels(args.labels, args.period)
    with rasterio.open(args.labels) as lb:
        ae_tf, profile = lb.transform, lb.profile.copy()
    with rasterio.open(args.forest_mask) as fm:
        assert (fm.width, fm.height, fm.transform) == (w, h, ae_tf), "산림 마스크 격자가 AlphaEarth 격자와 다름"
        forest = fm.read(1) == 1
    with rasterio.open(args.s2) as s:
        s2_tf, s2_h, s2_w = s.transform, s.height, s.width
    hold_s2 = block_split(s2_h, s2_w, args.block, args.seed)
    px_ha = abs(ae_tf.a * ae_tf.e) / 1e4

    # 1) 학습 (AE + S2 지수)
    idx, hold, r2, c2 = samples(labels_full, w, h, ae_tf, s2_tf, s2_h, s2_w, hold_s2, args.seed)
    embs = {}
    for y in AE_YEARS:
        with rasterio.open(ae_path(args.ae_dir, y)) as ds:
            embs[y] = ds.read().reshape(64, -1)[:, idx].T.copy()
    F_ae, n_ae, _, valid_s = ae_features(embs, y0, y1)
    del embs
    print("[S2] 전체 화소 지수 특징 계산 …", flush=True)
    s2_rows, s2_cols = np.divmod(np.arange(s2_h * s2_w), s2_w)
    F_s2_all, n_s2, s2_valid_all = s2_features(args.s2, s2_rows, s2_cols)  # [S2 화소, 48]
    del s2_rows, s2_cols
    s2_flat = r2 * s2_w + c2
    inside = (r2 >= 0) & (r2 < s2_h) & (c2 >= 0) & (c2 < s2_w)
    lab = {k: v.ravel()[idx] for k, v in labels_full.items()}
    lab["FC_TYPE"] = np.where(valid_s & s2_valid_all[s2_flat[idx]], lab["FC_TYPE"], 255)
    F = np.concatenate([F_ae, F_s2_all[s2_flat[idx]]], 1)
    clf, met = train_classifier(F, n_ae + n_s2, lab, hold[idx], args.seed)
    del F, F_ae
    print(f"[학습] AE+S2지수 특징 {len(n_ae) + len(n_s2)}개: macro F1 {met['macro_f1']}, 변화 AUROC {met['changed_auroc']}, "
          f"AP {met['changed_ap']} | 국유림 검증 AP {met['verified_changed_ap']}", flush=True)

    # 2) 산림 마스크 안 예측 (띠 단위)
    files = {y: rasterio.open(ae_path(args.ae_dir, y)) for y in AE_YEARS}
    raw = np.full((h, w), 255, np.uint8)
    prob = np.full((len(clf.classes_), h, w), np.nan, np.float32)
    year = np.zeros((h, w), np.uint16)
    in_p = [i for i, (a, b) in enumerate(zip(AE_YEARS, AE_YEARS[1:])) if y0 <= a and b <= y1]
    try:
        for r in range(0, h, args.strip):
            win = Window(0, r, w, min(args.strip, h - r))
            rows = slice(r, r + win.height)
            sel = forest[rows].ravel() & inside.reshape(h, w)[rows].ravel()
            if not sel.any():
                continue
            embs = {y: f.read(window=win).reshape(64, -1).T[sel] for y, f in files.items()}
            Fa, _, sc, ok = ae_features(embs, y0, y1)
            s2i = s2_flat.reshape(h, w)[rows].ravel()[sel]
            ok &= s2_valid_all[s2i]
            pr = clf.predict_proba(np.concatenate([Fa[ok], F_s2_all[s2i[ok]]], 1))
            pos = np.where(sel)[0][ok]
            for name, arr, vals in (("raw", raw, clf.classes_[pr.argmax(1)]), ("year", year, sc["ae_max_year"][ok])):
                flat = arr[rows].reshape(-1)
                flat[pos] = vals
                arr[rows] = flat.reshape(win.height, w)
            for k in range(len(clf.classes_)):
                flat = prob[k, rows].reshape(-1)
                flat[pos] = pr[:, k]
                prob[k, rows] = flat.reshape(win.height, w)
            if (r // args.strip) % 8 == 7:
                print(f"  [map] 행 {r + win.height}/{h} ({(time.time() - t0) / 60:.0f}분)", flush=True)
    finally:
        for f in files.values():
            f.close()
    del F_s2_all

    # 3) 클래스별 최소 면적
    valid = raw != 255
    final = class_mmu(raw, valid, px_ha, mmu)
    year = np.where(np.isin(final, [1, 2, 3]), year, 0).astype(np.uint16)

    # 4) 저장
    profile.update(count=1)
    write_tif(outdir / "final_change_class.tif", profile, {"change_class": final}, dtype="uint8", nodata=255)
    write_tif(outdir / "final_change_prob.tif", profile, {f"p_{FC_NAMES[k]}": prob[i] for i, k in enumerate(clf.classes_)})
    write_tif(outdir / "final_change_year.tif", profile, {"change_year": year}, dtype="uint16", nodata=0)
    import joblib
    joblib.dump({"clf": clf, "features": n_ae + n_s2, "classes": FC_NAMES, "period": args.period, "mmu_ha": mmu},
                outdir / "final_classifier.joblib")

    # 변화 패치 폴리곤
    import geopandas as gpd
    from shapely.geometry import shape
    recs = []
    for k in (1, 2, 3):
        comp, n = ndimage.label(final == k, structure=np.ones((3, 3)))
        if n == 0:
            continue
        ids = comp.ravel()
        area = np.bincount(ids, minlength=n + 1) * px_ha
        pk = np.nan_to_num(prob[list(clf.classes_).index(k)]).ravel()
        mean_p = np.bincount(ids, weights=pk, minlength=n + 1) / np.maximum(np.bincount(ids, minlength=n + 1), 1)
        yrs = np.zeros((n + 1, y1 - y0 + 1), np.int64)
        yv = year.ravel()
        m = (ids > 0) & (yv >= y0)
        np.add.at(yrs, (ids[m], yv[m] - y0), 1)
        for geom, cid in shapes(comp.astype(np.int32), mask=comp > 0, transform=ae_tf, connectivity=8):
            cid = int(cid)
            recs.append({"cid": cid, "class": k, "class_ko": CLASS_KO[k], "area_ha": round(area[cid], 3),
                         "mean_prob": round(float(mean_p[cid]), 3),
                         "change_year": int(y0 + yrs[cid].argmax()) if yrs[cid].sum() else None, "geometry": shape(geom)})
    patches = gpd.GeoDataFrame(recs, crs=profile["crs"])
    # 같은 연결요소가 여러 폴리곤으로 나올 수 있어 (클래스, 연결요소 번호)로 묶음
    patches = patches.dissolve(by=["class", "cid"], aggfunc="first", as_index=False).drop(columns="cid")
    patches.insert(0, "patch_id", [f"C{i:05d}" for i in range(len(patches))])
    patches.to_file(outdir / "final_change_patches.gpkg", layer="변화패치", driver="GPKG")

    # 홀드아웃 평가 (최종 지도)
    from sklearn.metrics import precision_recall_fscore_support
    labf = labels_full["FC_TYPE"]
    mm = hold.reshape(h, w) & (labf != 255) & (final != 255)
    pr_, rc_, f1_, _ = precision_recall_fscore_support(labf[mm], final[mm], labels=[0, 1, 2, 3], zero_division=0)
    area_ha = {CLASS_KO[k]: round(float((final == k).sum()) * px_ha, 1) for k in CLASS_KO}
    report = {"period": f"{y0}→{y1}", "model": "AlphaEarth V1 + Sentinel-2 지수 (HistGradientBoosting)",
              "forest_mask": Path(args.forest_mask).name, "mmu_ha": {CLASS_KO[k]: v for k, v in mmu.items()},
              "classifier_holdout": met, "final_map_holdout": {
                  "macro_f1": round(float(f1_.mean()), 4),
                  **{CLASS_KO[k]: {"precision": round(float(pr_[k]), 3), "recall": round(float(rc_[k]), 3),
                                   "f1": round(float(f1_[k]), 3)} for k in CLASS_KO}},
              "area_ha": area_ha, "patches": patches.groupby("class_ko").agg(
                  개수=("patch_id", "size"), 면적_ha=("area_ha", "sum"), 중앙값_ha=("area_ha", "median")).round(2).to_dict("index"),
              "change_year_area_ha": {CLASS_KO[k]: {int(yy): round(float(((final == k) & (year == yy)).sum()) * px_ha, 1)
                                                    for yy in range(y0 + 1, y1 + 1)} for k in (1, 2, 3)},
              "elapsed_min": round((time.time() - t0) / 60, 1)}
    (outdir / "final_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    # 미리보기
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    step = 4
    view = final[::step, ::step].astype(float)
    view[view == 255] = np.nan
    fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
    ax.imshow(view, cmap=ListedColormap(["#d9ead3", "#d7301f", "#7b3294", "#1a9850"]), vmin=-0.5, vmax=3.5,
              interpolation="nearest")
    ax.set_axis_off()
    ax.set_title(f"Chuncheon forest change {y0}-{y1} (AlphaEarth+S2, forest mask, MMU)", fontsize=11)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#d9ead3", "#d7301f", "#7b3294", "#1a9850"]]
    ax.legend(handles, [f"no change", f"harvest {area_ha['벌채·갱신']:,.0f} ha", f"conversion {area_ha['산지전용']:,.0f} ha",
                        f"gain {area_ha['산림증가']:,.0f} ha"], loc="lower right", fontsize=9)
    fig.savefig(outdir / "final_preview.png", bbox_inches="tight")

    print(json.dumps({k: report[k] for k in ("final_map_holdout", "area_ha", "patches", "change_year_area_ha")},
                     indent=1, ensure_ascii=False))
    print(f"[완료] {report['elapsed_min']}분, 결과: {outdir}")


if __name__ == "__main__":
    main()


In [7]:
# [08-13] 최종 변화지도 (AE+S2지수, 산림 마스크, 최소면적 벌채·증가 0.5 ha / 전용 0.2 ha) → Drive/GEE_Export/final_change_2019_2025
import glob, shutil
s2_54 = sorted(glob.glob('/content/data/Chuncheon_S2_2017_2025_54Bands*.tif'))[0]
%run step08_final_change_map.py --ae-dir /content/data/ae --labels /content/data/Chuncheon_labels_AE_10m.tif --s2 {s2_54} --forest-mask /content/data/Chuncheon_forestmask_AE_10m.tif --outdir /content/runs/final_p2
shutil.copytree('/content/runs/final_p2', '/content/drive/MyDrive/GEE_Export/final_change_2019_2025', dirs_exist_ok=True)
print('Drive 복사 완료: GEE_Export/final_change_2019_2025')

[S2] 전체 화소 지수 특징 계산 …
[학습] AE+S2지수 특징 250개: macro F1 0.5853, 변화 AUROC 0.9918, AP 0.8792 | 국유림 검증 AP 0.9934
  [map] 행 1024/4549 (21분)
  [map] 행 2048/4549 (35분)
  [map] 행 3072/4549 (47분)
  [map] 행 4096/4549 (57분)
{
 "final_map_holdout": {
  "macro_f1": 0.6056,
  "무변화": {
   "precision": 0.996,
   "recall": 0.99,
   "f1": 0.993
  },
  "벌채·갱신": {
   "precision": 0.745,
   "recall": 0.963,
   "f1": 0.84
  },
  "산지전용": {
   "precision": 0.132,
   "recall": 0.435,
   "f1": 0.202
  },
  "산림증가": {
   "precision": 0.411,
   "recall": 0.366,
   "f1": 0.387
  }
 },
 "area_ha": {
  "무변화": 81111.1,
  "벌채·갱신": 2287.6,
  "산지전용": 662.8,
  "산림증가": 978.0
 },
 "patches": {
  "벌채·갱신": {
   "개수": 719,
   "면적_ha": 2287.64,
   "중앙값_ha": 0.95
  },
  "산림증가": {
   "개수": 566,
   "면적_ha": 977.95,
   "중앙값_ha": 0.73
  },
  "산지전용": {
   "개수": 1620,
   "면적_ha": 662.83,
   "중앙값_ha": 0.27
  }
 },
 "change_year_area_ha": {
  "벌채·갱신": {
   "2020": 404.8,
   "2021": 359.3,
   "2022": 448.1,
   "2023": 612.8,
   "2024": 211

In [9]:
# [08-14] 산지전용 고시 라벨 업로드: C:\src\ForestChangeGeoAI\output\산지전용\Chuncheon_convlabels_AE_10m.tif 선택
#         (밴드 이름 CONV_LABEL 로 올바른 파일인지 확인 — 다른 파일이면 지우고 다시 업로드)
import os, shutil, rasterio
from google.colab import files
dst = '/content/data/Chuncheon_convlabels_AE_10m.tif'
G = '/content/drive/MyDrive/GEE_Export/Chuncheon_convlabels_AE_10m.tif'
def is_conv(p):
    try:
        with rasterio.open(p) as ds:
            return ds.descriptions[0] == 'CONV_LABEL'
    except Exception:
        return False
for p in (dst, G):
    if os.path.exists(p) and not is_conv(p):
        os.remove(p); print('잘못된 파일 삭제:', p)
if not os.path.exists(dst):
    if os.path.exists(G):
        shutil.copy(G, dst)
    else:
        for fn in files.upload():
            if is_conv(fn):
                shutil.move(fn, dst); shutil.copy(dst, G)
            else:
                os.remove(fn); print('산지전용 라벨이 아님(CONV_LABEL 밴드 없음):', fn)
print('준비:', is_conv(dst) if os.path.exists(dst) else False)

잘못된 파일 삭제: /content/data/Chuncheon_convlabels_AE_10m.tif
잘못된 파일 삭제: /content/drive/MyDrive/GEE_Export/Chuncheon_convlabels_AE_10m.tif


Saving Chuncheon_convlabels_AE_10m.tif to Chuncheon_convlabels_AE_10m.tif
준비: True


In [ ]:
%%writefile step08_conv_label_experiment.py
#!/usr/bin/env python3
# coding: utf-8
# [colab 정리본] 원본: s2_tsfm/conv_label_experiment.py — 파일명·모듈명만 실험 단계 번호에 맞춤 (계산 로직 동일)
"""산지전용 고시 라벨(Chuncheon_convlabels_AE_10m.tif)로 ① 최종 변화지도 검증 ② 라벨 보강 재학습 비교.

① 검증: final_change_class.tif 에서 고시 라벨 화소·필지가 산지전용(2)/변화(1~3)로 분류된 비율, 무작위 산림 화소와 비교
② 재학습: 같은 표본·홀드아웃(step08_final_change_map.samples)으로 AE+S2지수 분류기를 두 번 학습
   기본  : 합의 라벨 TRAIN_P2
   보강  : + 전용확정(1) 화소 → 산지전용(2), 전용이전(3) 중 라벨 없는 화소 → 무변화(0) (2019 이전 전용, 이후 안정)
           학습 블록의 고시 라벨 화소는 표본에 모두 포함
   비교  : 홀드아웃 블록 산지전용 정밀도·재현율·F1, macro F1, 홀드아웃 전용확정 화소 재현율

Colab:
  %run step08_conv_label_experiment.py --final /content/runs/final_p2/final_change_class.tif --conv /content/data/Chuncheon_convlabels_AE_10m.tif \
       --ae-dir /content/data/ae --labels /content/data/Chuncheon_labels_AE_10m.tif --s2 {s2_54} --outdir /content/runs/conv_exp
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import rasterio
from sklearn.metrics import precision_recall_fscore_support

from step08_ae_change_classifier import AE_YEARS, PERIOD_YEARS, ae_features, ae_path, s2_features
from step08_final_change_map import samples
from step05_s2_tsfm import FC_NAMES, block_split, read_labels, train_classifier

CONV = {1: "전용확정", 2: "전용후보", 3: "전용이전"}


def validate_final(final_path, conv, forest_hold=None):
    with rasterio.open(final_path) as ds:
        fin = ds.read(1)
    out = {}
    rng = np.random.default_rng(0)
    forest = np.flatnonzero(fin.ravel() != 255)
    base = fin.ravel()[rng.choice(forest, 200000, replace=False)]
    out["무작위_산림화소"] = {"산지전용": round(float((base == 2).mean()), 4), "변화(1~3)": round(float(np.isin(base, [1, 2, 3]).mean()), 4)}
    for k, name in CONV.items():
        m = conv == k
        v = fin[m]
        inside = v != 255
        out[name] = {"화소": int(m.sum()), "산림마스크_안": round(float(inside.mean()), 3),
                     "산지전용_비율": round(float((v[inside] == 2).mean()), 3) if inside.any() else None,
                     "변화(1~3)_비율": round(float(np.isin(v[inside], [1, 2, 3]).mean()), 3) if inside.any() else None,
                     "벌채·갱신_비율": round(float((v[inside] == 1).mean()), 3) if inside.any() else None}
    return out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--final", required=True)
    p.add_argument("--conv", required=True)
    p.add_argument("--ae-dir", required=True)
    p.add_argument("--labels", required=True)
    p.add_argument("--s2", required=True)
    p.add_argument("--outdir", default="runs/conv_exp")
    p.add_argument("--seed", type=int, default=42)
    args, _ = p.parse_known_args()
    out = Path(args.outdir)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    labels_full, (w, h), _ = read_labels(args.labels, "P2")
    with rasterio.open(args.labels) as lb:
        ae_tf = lb.transform
    with rasterio.open(args.conv) as cv:
        assert (cv.width, cv.height, cv.transform) == (w, h, ae_tf), "고시 라벨 격자가 AlphaEarth 격자와 다름"
        conv = cv.read(1)

    # ① 최종 지도 검증
    val = validate_final(args.final, conv)
    print("[① 최종 지도 검증]")
    for k, v in val.items():
        print(f"  {k}: {v}")

    # ② 라벨 보강 재학습
    with rasterio.open(args.s2) as s:
        s2_tf, s2_h, s2_w = s.transform, s.height, s.width
    hold_s2 = block_split(s2_h, s2_w, 512, args.seed)
    base_fc = labels_full["FC_TYPE"]
    aug_fc = base_fc.copy()
    aug_fc[conv == 1] = 2
    aug_fc[(conv == 3) & (base_fc == 255)] = 0
    variants = {"기본(합의 라벨)": base_fc, "보강(+고시 산지전용)": aug_fc}

    idx_sets, hold = {}, None
    for name, fc in variants.items():
        lab = dict(labels_full, FC_TYPE=fc)
        idx, hold, r2, c2 = samples(lab, w, h, ae_tf, s2_tf, s2_h, s2_w, hold_s2, args.seed)
        if name.startswith("보강"):  # 학습 블록의 고시 라벨 화소는 모두 포함
            extra = np.flatnonzero(~hold & np.isin(conv.ravel(), [1, 3]) & (fc.ravel() != 255))
            idx = np.union1d(idx, extra)
        idx_sets[name] = idx
    all_idx = np.union1d(*idx_sets.values())
    print(f"[② 표본] 기본 {len(idx_sets['기본(합의 라벨)']):,} / 보강 {len(idx_sets['보강(+고시 산지전용)']):,} 화소, "
          f"보강으로 추가된 산지전용 학습화소 {int((~hold & (conv.ravel() == 1)).sum())}, 홀드아웃 전용확정 화소 {int((hold & (conv.ravel() == 1)).sum())}")

    embs = {}
    for y in AE_YEARS:
        with rasterio.open(ae_path(args.ae_dir, y)) as ds:
            embs[y] = ds.read().reshape(64, -1)[:, all_idx].T.copy()
    F_ae, n_ae, _, valid_ae = ae_features(embs, *PERIOD_YEARS["P2"])
    del embs
    s2_flat = r2 * s2_w + c2
    F_s2, n_s2, valid_s2 = s2_features(args.s2, r2[all_idx], c2[all_idx])
    F_all = np.concatenate([F_ae, F_s2], 1)
    ok_all = valid_ae & valid_s2

    results = {}
    for name, fc in variants.items():
        sel = np.searchsorted(all_idx, idx_sets[name])
        F = F_all[sel]
        lab = {k: v.ravel()[idx_sets[name]] for k, v in labels_full.items()}
        lab["FC_TYPE"] = np.where(ok_all[sel], fc.ravel()[idx_sets[name]], 255)
        clf, met = train_classifier(F, n_ae + n_s2, lab, hold[idx_sets[name]], args.seed, max_per_class=60000)
        # 같은 평가 기준: 홀드아웃 블록의 '기본' 라벨 + 홀드아웃 고시 전용확정 화소
        te = np.searchsorted(all_idx, np.intersect1d(all_idx, np.flatnonzero(hold)))
        te = te[ok_all[te]]
        pred = clf.predict(F_all[te])
        y_base = base_fc.ravel()[all_idx[te]]
        m = y_base != 255
        pr, rc, f1, _ = precision_recall_fscore_support(y_base[m], pred[m], labels=[0, 1, 2, 3], zero_division=0)
        cm = conv.ravel()[all_idx[te]] == 1
        res = {"macro_F1(기본라벨)": round(float(f1.mean()), 4),
               "산지전용_정밀도": round(float(pr[2]), 3), "산지전용_재현율": round(float(rc[2]), 3), "산지전용_F1": round(float(f1[2]), 3),
               "벌채갱신_F1": round(float(f1[1]), 3), "산림증가_F1": round(float(f1[3]), 3),
               "홀드아웃_전용확정화소": int(cm.sum()),
               "전용확정_산지전용예측비율": round(float((pred[cm] == 2).mean()), 3) if cm.any() else None,
               "전용확정_변화예측비율": round(float(np.isin(pred[cm], [1, 2, 3]).mean()), 3) if cm.any() else None,
               "분류기_자체지표": {k: met[k] for k in ("macro_f1", "changed_auroc", "changed_ap", "n_train")}}
        results[name] = res
        print(f"[② {name}] {json.dumps(res, ensure_ascii=False)}", flush=True)

    report = {"검증_최종지도": val, "재학습_비교": results, "elapsed_min": round((time.time() - t0) / 60, 1)}
    (out / "conv_experiment.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[완료] {report['elapsed_min']}분 → {out / 'conv_experiment.json'}")


if __name__ == "__main__":
    main()


In [11]:
# [08-15] ① 최종 지도 검증 + ② 고시 라벨 보강 재학습 비교 → Drive/GEE_Export/compare_runs/conv_exp
%run step08_conv_label_experiment.py --final /content/runs/final_p2/final_change_class.tif --conv /content/data/Chuncheon_convlabels_AE_10m.tif --ae-dir /content/data/ae --labels /content/data/Chuncheon_labels_AE_10m.tif --s2 {s2_54} --outdir /content/runs/conv_exp
shutil.copytree('/content/runs/conv_exp', '/content/drive/MyDrive/GEE_Export/compare_runs/conv_exp', dirs_exist_ok=True)

[① 최종 지도 검증]
  무작위_산림화소: {'산지전용': 0.0076, '변화(1~3)': 0.0467}
  전용확정: {'화소': 207, '산림마스크_안': 0.889, '산지전용_비율': 0.489, '변화(1~3)_비율': 0.804, '벌채·갱신_비율': 0.315}
  전용후보: {'화소': 7301, '산림마스크_안': 0.455, '산지전용_비율': 0.11, '변화(1~3)_비율': 0.135, '벌채·갱신_비율': 0.007}
  전용이전: {'화소': 28914, '산림마스크_안': 0.721, '산지전용_비율': 0.005, '변화(1~3)_비율': 0.019, '벌채·갱신_비율': 0.002}
[② 표본] 기본 910,627 / 보강 936,725 화소, 보강으로 추가된 산지전용 학습화소 207, 홀드아웃 전용확정 화소 0
[② 기본(합의 라벨)] {"macro_F1(기본라벨)": 0.5853, "산지전용_정밀도": 0.096, "산지전용_재현율": 0.548, "산지전용_F1": 0.163, "벌채갱신_F1": 0.823, "산림증가_F1": 0.364, "홀드아웃_전용확정화소": 0, "전용확정_산지전용예측비율": null, "전용확정_변화예측비율": null, "분류기_자체지표": {"macro_f1": 0.5853, "changed_auroc": 0.9918, "changed_ap": 0.8792, "n_train": 136462}}
[② 보강(+고시 산지전용)] {"macro_F1(기본라벨)": 0.5899, "산지전용_정밀도": 0.115, "산지전용_재현율": 0.536, "산지전용_F1": 0.189, "벌채갱신_F1": 0.828, "산림증가_F1": 0.351, "홀드아웃_전용확정화소": 0, "전용확정_산지전용예측비율": null, "전용확정_변화예측비율": null, "분류기_자체지표": {"macro_f1": 0.588, "changed_auroc": 0.9924, "changed_ap": 0.8826, "n_

'/content/drive/MyDrive/GEE_Export/compare_runs/conv_exp'